# CarDD Vehicle Damage Detection — Full Step-by-Step Pipeline

Built to follow the project proposal objective by objective, on the **real CarDD dataset**.

**Part A — Data exploration & cleaning (Objective 2.2)** — Steps 1-9
**Part B — Preprocessing: resize/normalise, split, augment, convert (Objective 2.2)** — Steps 10-15
**Part C — Train the five models (Objectives 2.3, 2.4)** — YOLOv8, Faster R-CNN, SSD, EfficientDet/RetinaNet, ResNet50
**Part D — Evaluate & compare (Objective 2.5)** — mAP, Precision, Recall, F1, inference time, confusion matrices
**Part E — Explainability (Objective 2.5)** — Grad-CAM + EigenCAM
**Part F — Inference demo**

---
**Run order (Kaggle — original platform, for reference):** attach the CarDD dataset as an Input, set **Settings → Accelerator → GPU P100** (or T4 x2) and **Settings → Internet → On** (needed for `pip install` and pretrained weights), then run top to bottom — Step 0 auto-detects the dataset under `/kaggle/input`. For the final run use **Save Version → Save & Run All (Commit)** so it trains in the background and the checkpoints in `/kaggle/working` are saved with the notebook version. GPU needed for Parts C-F.

**Viva note:** every step has a short *why* in its heading. Read them - you will be asked to justify these choices, and the marks are in the justification, not the code running.


---

**Run order (Google Colab):** (1) **Runtime -> Change runtime type -> T4 GPU -> Save**. (2) Run the **Mount Google Drive** cell and click *Allow*. (3) Make sure your CarDD folder (with `annotations/`, `train2017/`, `val2017/`, `test2017/`) is in your Drive — the paths cell auto-finds it anywhere under `MyDrive`. (4) Run top to bottom. Checkpoints save to `MyDrive/cardd_ckpts`, so if the runtime disconnects you can just re-run from the top and it resumes from Drive instead of starting over. GPU is needed for Parts C–F.

### Step 0 — Environment & paths

Install dependencies, import libraries, and point `DATA_ROOT` at your `CarDD_COCO` folder (containing `annotations/` and the `*2017/` image folders).


In [ ]:
# Install dependencies — idempotent & self-healing.
# If the kernel is ever reset/restarted mid-session (idle timeout, crash, reconnect),
# this specific cell may show as "not run" (or only partially run) even though every
# OTHER cell still shows old outputs from before the reset. Any cell that imports ultralytics/torchmetrics/pytorch_grad_cam then fails with
# ModuleNotFoundError, because the fresh runtime never got these packages.
#
# Fix: define ensure_packages() here, and ALSO call it again at the top of every cell
# that needs one of these libraries (C1 YOLO, C6 RT-DETR, evaluator, Grad-CAM/EigenCAM).
# That way those cells self-install what they need even if this cell was skipped,
# never finished, or ran in a runtime that has since been reset.
import importlib, subprocess, sys

def ensure_packages(pip_specs, import_checks=None):
    # Install any pip_specs whose matching module isn't importable yet.
    # pip_specs: list of pip package strings, e.g. ['ultralytics', 'torchmetrics']
    # import_checks: optional list of the actual module names to test import for
    #                (defaults to pip_specs, when the import name == package name)
    import_checks = import_checks or pip_specs
    missing = []
    for pkg, mod in zip(pip_specs, import_checks):
        try:
            importlib.import_module(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f'Installing missing packages: {missing}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
        print('Done.')
    else:
        print('All required packages already present:', pip_specs)

# Core packages needed throughout the notebook.
try:
    ensure_packages(
        ['ultralytics', 'torchmetrics', 'grad-cam', 'pycocotools', 'pyyaml', 'seaborn', 'scikit-learn'],
        import_checks=['ultralytics', 'torchmetrics', 'pytorch_grad_cam', 'pycocotools', 'yaml', 'seaborn', 'sklearn']
    )
except subprocess.CalledProcessError:
    raise RuntimeError('pip install failed. On Colab this is usually a transient network hiccup '
                       'or a version conflict. Just re-run this cell; if it persists, do '
                       'Runtime -> Restart runtime, then Run all.')

# EfficientDet via the effdet library (Ross Wightman). effdet 0.4.1 is the last stable
# release and needs a compatible timm; pin both so the C4 EfficientDet path is reproducible.
# If install fails (e.g. version conflict with ultralytics' timm), C4 falls back to RetinaNet.
try:
    ensure_packages(["effdet==0.4.1", "timm>=0.9.2"], import_checks=["effdet", "timm"])
except subprocess.CalledProcessError:
    print("effdet not installed - EfficientDet step uses RetinaNet stand-in")


In [ ]:
# Imports
import os, json, shutil, glob, time, warnings, copy
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image, ImageEnhance
import torch
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Parts A/B run on CPU; training (C-F) needs a GPU.')


### Step 0.1 — Mount Google Drive  (Colab only)
Run this first. A popup asks you to pick your Google account and click **Allow**. Your Drive then appears under `/content/drive/MyDrive/`. This is where your CarDD dataset lives and where trained models will be saved.

In [ ]:
# Mount your Google Drive so the notebook can read the CarDD dataset
# and save trained model checkpoints that survive runtime disconnects.
from google.colab import drive
drive.mount('/content/drive')

# Quick check: list the top level of your Drive so you can see your CarDD folder.
import os
print('Top-level folders in your Drive:')
for item in sorted(os.listdir('/content/drive/MyDrive'))[:40]:
    print('  ', item)

In [ ]:
# ============================================================================
#  Paths — get the FULL CarDD_COCO dataset out of Google Drive, reliably.
#
#  Why this cell is written this way: uploading thousands of LOOSE image files
#  to Drive is unreliable (that is exactly how train2017 ended up with 1 file).
#  A SINGLE ZIP uploads reliably. So this cell tries, in order:
#    1) ZIP MODE  — find a CarDD*.zip anywhere obvious in your Drive, copy that
#       ONE file to fast local disk, extract ALL of it, and verify every single
#       zip entry landed on disk.
#    2) FOLDER MODE — fall back to the CarDD_COCO folder in Drive (works now
#       that your folder upload finally completed) and copy it locally.
#  Then, whichever mode ran, it does a NAME-BY-NAME completeness check:
#  every image the annotation files reference must exist on disk — or it STOPS
#  with exact instructions, BEFORE any GPU time is spent.
#
#  Defines the same names every later cell uses: DATA_ROOT, YOLO_ROOT,
#  CLEAN_ANN_DIR, ANN_DIR, SPLIT_MAP, IMG_DIRS, CKPT_ROOT. Nothing downstream changes.
# ============================================================================
import os, glob, json, subprocess, shutil, zipfile

DRIVE = '/content/drive/MyDrive'

# Optional manual override: if you know exactly where your zip is, set it here, e.g.
# ZIP_PATH_OVERRIDE = '/content/drive/MyDrive/CarDD_release.zip'
ZIP_PATH_OVERRIDE = None

# ---------------------------------------------------------------- find a zip
zips = [] if ZIP_PATH_OVERRIDE else sum((glob.glob(p) for p in [
    os.path.join(DRIVE, '*.zip'),
    os.path.join(DRIVE, 'CarDD*', '*.zip'),
    os.path.join(DRIVE, 'CarDD*', '*', '*.zip'),
]), [])
zips = [z for z in zips if 'cardd' in os.path.basename(z).lower()]
ZIP_PATH = ZIP_PATH_OVERRIDE or (max(zips, key=os.path.getsize) if zips else None)

EXTRACT_BASE = '/content/CarDD_extracted'

def find_cardd_root(base):
    """Folder whose annotations/ contains instances_train2017.json, however nested."""
    if not os.path.isdir(base):
        return None
    for root, dirs, files in os.walk(base):
        if os.path.basename(root) == 'annotations' and 'instances_train2017.json' in files:
            return os.path.dirname(root)
    return None

DATA_ROOT = None

if ZIP_PATH and os.path.exists(ZIP_PATH):
    # ------------------------------------------------------------- ZIP MODE
    sz_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f'ZIP MODE — found dataset zip in Drive: {ZIP_PATH}  ({sz_mb:.0f} MB)')
    marker = os.path.join(EXTRACT_BASE, '.extract_complete')
    if os.path.exists(marker) and find_cardd_root(EXTRACT_BASE):
        print('Already fully extracted earlier this session — skipping extraction.')
    else:
        shutil.rmtree(EXTRACT_BASE, ignore_errors=True)
        os.makedirs(EXTRACT_BASE, exist_ok=True)
        # Copy the ONE zip file from Drive to fast local disk first (a single big
        # file copies quickly and completely; extracting straight from the Drive
        # mount is slow and flaky).
        local_zip = '/content/_cardd_dataset.zip'
        if not (os.path.exists(local_zip) and os.path.getsize(local_zip) == os.path.getsize(ZIP_PATH)):
            print('Copying zip from Drive to local disk...')
            shutil.copyfile(ZIP_PATH, local_zip)
        print('Extracting EVERYTHING from the zip...')
        with zipfile.ZipFile(local_zip) as zf:
            names = zf.namelist()
            total = len(names)
            for k, n in enumerate(names, 1):
                zf.extract(n, EXTRACT_BASE)
                if k % 1000 == 0 or k == total:
                    print(f'   {k}/{total} entries extracted', flush=True)
            # Verify EVERY file entry from the zip now exists on disk.
            missing_entries = [n for n in names
                               if not n.endswith('/')
                               and not os.path.exists(os.path.join(EXTRACT_BASE, n))]
        if missing_entries:
            raise RuntimeError(
                f'Extraction incomplete: {len(missing_entries)} entries did not land on disk, '
                f'e.g. {missing_entries[:3]}. Check local disk space (!df -h /content) and re-run.')
        open(marker, 'w').close()
        print(f'Extraction complete and VERIFIED: all {total} zip entries are on disk.')
    DATA_ROOT = find_cardd_root(EXTRACT_BASE)
    assert DATA_ROOT, ('Zip extracted, but no annotations/instances_train2017.json found inside it. '
                       'Make sure the zip contains the CarDD_COCO folder (annotations/ + train2017/ etc).')

else:
    # ---------------------------------------------------------- FOLDER MODE
    print('No CarDD zip found in Drive — falling back to FOLDER MODE.')
    SRC_DATA_ROOT = find_cardd_root(DRIVE)
    if SRC_DATA_ROOT is None:
        raise FileNotFoundError(
            'Could not find CarDD data in your Drive at all.\n'
            'BEST FIX (reliable): on your computer, zip the whole CarDD_release folder into\n'
            'a single file named CarDD_release.zip, upload that ONE file to My Drive, then\n'
            're-run this cell — it will find and extract it automatically.')
    print('Found CarDD folder at:', SRC_DATA_ROOT)
    DATA_ROOT = '/content/CarDD_COCO'
    SPLIT_SUBDIRS = ['train2017', 'val2017', 'test2017']
    src_counts = {s: (len(os.listdir(os.path.join(SRC_DATA_ROOT, s)))
                      if os.path.isdir(os.path.join(SRC_DATA_ROOT, s)) else -1)
                  for s in SPLIT_SUBDIRS}
    print('Drive folder file counts:', src_counts)
    need_copy = not os.path.exists(os.path.join(DATA_ROOT, 'annotations', 'instances_train2017.json'))
    if not need_copy:
        lc = {s: (len(os.listdir(os.path.join(DATA_ROOT, s)))
                  if os.path.isdir(os.path.join(DATA_ROOT, s)) else 0) for s in SPLIT_SUBDIRS}
        if any(lc[s] != src_counts.get(s, -1) for s in SPLIT_SUBDIRS):
            print('Local copy does not match Drive -> re-copying.')
            need_copy = True
        else:
            print('Local copy already present and matches Drive - skipping copy.')
    if need_copy:
        print(f'Copying folder from Drive to local disk ({DATA_ROOT})...')
        os.makedirs(DATA_ROOT, exist_ok=True)
        r = subprocess.run(['rsync', '-a', f'{SRC_DATA_ROOT}/', f'{DATA_ROOT}/'],
                           capture_output=True, text=True)
        if r.returncode != 0:
            shutil.rmtree(DATA_ROOT, ignore_errors=True)
            shutil.copytree(SRC_DATA_ROOT, DATA_ROOT)
        print('Copy done.')

# ------------------------------------------------ shared names for all cells
YOLO_ROOT     = '/content/CarDD_YOLO'
CLEAN_ANN_DIR = '/content/CarDD_clean/annotations'
ANN_DIR = os.path.join(DATA_ROOT, 'annotations')
SPLIT_MAP = {
    'train': ('instances_train2017.json', 'train2017'),
    'val':   ('instances_val2017.json',   'val2017'),
    'test':  ('instances_test2017.json',  'test2017'),
}
IMG_DIRS = {s: os.path.join(DATA_ROOT, sub) for s, (_, sub) in SPLIT_MAP.items()}
os.makedirs(CLEAN_ANN_DIR, exist_ok=True)
print('DATA_ROOT (local) =', DATA_ROOT)

# ---------------------------------------- FINAL NAME-BY-NAME COMPLETENESS CHECK
# Counting files can lie; this cannot: every image filename the annotation JSON
# references must actually exist on disk. If not, STOP before wasting GPU time.
print('\nFINAL COMPLETENESS CHECK (annotation names vs files on disk)')
print('-' * 60)
problems = []
for split, (ann_name, sub) in SPLIT_MAP.items():
    ann_path = os.path.join(ANN_DIR, ann_name)
    if not os.path.exists(ann_path):
        problems.append((split, 'annotation file missing entirely', 0, 0)); continue
    with open(ann_path) as f:
        names = [im['file_name'] for im in json.load(f)['images']]
    have = set(os.listdir(IMG_DIRS[split])) if os.path.isdir(IMG_DIRS[split]) else set()
    missing = [n for n in names if n not in have]
    print(f'   {split:6s}: {len(names)-len(missing):>5d} / {len(names):<5d} annotated images present'
          + ('' if not missing else f'   *** {len(missing)} MISSING ***'))
    if len(missing) > max(2, int(0.02 * len(names))):
        problems.append((split, sub, len(missing), len(names)))
print('-' * 60)
if problems:
    raise RuntimeError(
        'DATASET INCOMPLETE — stopping BEFORE training so no GPU time is wasted.\n'
        + '\n'.join(f'  - {p[0]}: {p[2]} of {p[3]} images missing from {p[1]}/' for p in problems)
        + '\n\nRELIABLE FIX: on your computer, zip the whole CarDD_release folder into ONE file\n'
          'named CarDD_release.zip, upload that single zip to My Drive (one file uploads\n'
          'reliably; thousands of loose files do not), then re-run this cell — it will\n'
          'automatically find the zip and extract everything.')
print('ALL SPLITS COMPLETE: every annotated image exists on disk. Safe to train.')

# ----------------------------------------------- persistent checkpoints (Drive)
CKPT_ROOT = '/content/drive/MyDrive/cardd_ckpts'
os.makedirs(CKPT_ROOT, exist_ok=True)
print('\nCheckpoints will be saved to (persisted on your Google Drive):', CKPT_ROOT)


### Step 0.3 — One-time checkpoint cleanup
The earlier broken run saved junk model weights to Drive. Leave `WIPE_OLD_CHECKPOINTS = True` for your **first** full run so everything trains fresh; set it to `False` afterwards so later sessions can resume instead of retraining.

In [ ]:
# ============================================================================
#  ONE-TIME CLEANUP — wipe checkpoints saved by the earlier BROKEN (1-image) run.
#  That run saved 'finished' weights for EVERY model to your Drive. They are
#  garbage (trained on 1 image), and if left in place, every training cell will
#  say 'checkpoint found -> skipping' and silently reuse them. That is exactly
#  the 'one model did not run' problem you kept hitting.
#
#  HOW TO USE:
#    - FIRST run after fixing the dataset:  leave WIPE_OLD_CHECKPOINTS = False   # <- set False so Run-all NEVER deletes your finished models
#      -> old junk is deleted, every model trains fresh on the FULL data.
#    - AFTER that run finishes:  change it to False
#      -> later sessions can skip/resume from the new, GOOD checkpoints.
# ============================================================================
WIPE_OLD_CHECKPOINTS = False   # <- set False so Run-all NEVER deletes your finished models

import os, shutil
if WIPE_OLD_CHECKPOINTS and os.path.isdir(CKPT_ROOT) and os.listdir(CKPT_ROOT):
    shutil.rmtree(CKPT_ROOT)
    os.makedirs(CKPT_ROOT, exist_ok=True)
    print('Old checkpoints WIPED from', CKPT_ROOT)
    print('-> every model will now train fresh on the full dataset.')
    print('-> after this run finishes, set WIPE_OLD_CHECKPOINTS = False (top of this cell).')
else:
    print('Keeping existing checkpoints - finished models will skip, interrupted ones resume.')


### Step 0.5 — Verify the real CarDD dataset is reachable

On Kaggle the dataset is attached as a **read-only Input** under `your Google Drive/`. The cell above auto-locates it (by finding `annotations/instances_train2017.json`), sanity-checks the file counts, then **copies it once to fast local writable disk** (`/content (local disk)/CarDD_COCO`) and points `DATA_ROOT` there. This is a one-time cost per session and makes every later step (integrity audit, cleaning, YOLO conversion, training) far faster than reading from the input mount. If the session restarts, just re-run the cell — it detects and verifies an existing copy before deciding whether to re-copy.


In [ ]:
# Verify the real CarDD data is present (no synthetic fallback)
missing = [os.path.join(ANN_DIR, a) for a, _ in SPLIT_MAP.values()
           if not os.path.exists(os.path.join(ANN_DIR, a))]
assert not missing, (
    'Real CarDD annotation files not found:\n  ' + '\n  '.join(missing) +
    '\n\nCheck that your CarDD folder is in your Google Drive and the Drive-mount cell ran. '
    'If auto-detect failed, confirm the folder contains annotations/instances_train2017.json, '
    'and set DRIVE_SEARCH_BASE (in the paths cell) to point at it.'
)
print('Real CarDD found at', DATA_ROOT, '-> using it.')


## Part A — Data exploration & cleaning  (Objective 2.2)

### Step 1 — Inspect the dataset on disk
*Why:* confirm folder structure and image counts before trusting anything downstream.


In [ ]:
print('DATA_ROOT contents:'); print('='*50)
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root.replace(DATA_ROOT, '').count(os.sep)
    if depth > 1: continue
    print('  ' * depth + os.path.basename(root) + '/  (' + str(len(files)) + ' files)')
print('\nImage counts per split folder:')
for split, d in IMG_DIRS.items():
    n = len(glob.glob(os.path.join(d, '*'))) if os.path.isdir(d) else 0
    print(f'   {split:<6}: {n} images   ({d})')


### Step 2 — Load COCO annotations and read the real class list
*Why:* the 0-based class order set here is used everywhere downstream; lock it in once.


In [ ]:
from pycocotools.coco import COCO
coco = {}
for split, (ann, _) in SPLIT_MAP.items():
    p = os.path.join(ANN_DIR, ann)
    if os.path.exists(p):
        coco[split] = COCO(p)

# --- Build the class list from category NAMES, and a SEPARATE id->index map PER SPLIT ---
# Bug this fixes: train/val/test are three separate COCO export files. Nothing guarantees
# they assign the same numeric category_id to the same class name (e.g. train might say
# id 3 = "crack" while val says id 3 = "glass shatter"). The previous version built ONE
# global id->index table from train's categories only, then reused it for val/test
# annotations too - silently mislabeling every split whose ids don't match train's
# numbering. Symptom: every detector architecture scores near-zero mAP on most classes
# while training loss looks completely healthy - the model IS learning, it's just being
# trained/evaluated against systematically wrong class labels for most classes.
print('Category id -> name, per split (these should all match - flagged below if not):')
per_split_cats = {}
all_names = set()
for split, c in coco.items():
    cats_this_split = sorted(c.loadCats(c.getCatIds()), key=lambda x: x['id'])
    per_split_cats[split] = cats_this_split
    all_names.update(cc['name'] for cc in cats_this_split)
    print(f'  {split}:', [(cc['id'], cc['name']) for cc in cats_this_split])

_id_to_name_per_split = {s: {cc['id']: cc['name'] for cc in cs} for s, cs in per_split_cats.items()}
_splits = list(_id_to_name_per_split)
for s in _splits[1:]:
    if _id_to_name_per_split[s] != _id_to_name_per_split[_splits[0]]:
        print(f'  *** WARNING: "{s}" category id->name mapping DIFFERS from "{_splits[0]}". ***')
        print(f'  *** This is exactly the bug that caused near-zero mAP on most classes. ***')
        print(f'  *** The per-split mapping below corrects for it automatically. ***')

CLASS_NAMES = sorted(all_names)
NUM_CLASSES = len(CLASS_NAMES)
name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}
print('Damage classes (0-based order used throughout, derived from class NAMES):')
for i, n in enumerate(CLASS_NAMES):
    print(f'   {i}: {n}')

# Per-split category_id -> canonical index. Every downstream cell (YOLO conversion,
# CocoDetDataset, CocoMultiLabel) must key off THIS - by split - not one shared dict.
catid_to_idx_by_split = {
    split: {cc['id']: name_to_idx[cc['name']] for cc in cats_this_split}
    for split, cats_this_split in per_split_cats.items()
}
catid_to_contig_by_split = {
    split: {cid: idx + 1 for cid, idx in d.items()}
    for split, d in catid_to_idx_by_split.items()
}
CAT_ID_TO_NAME = {cc['id']: cc['name'] for cs in per_split_cats.values() for cc in cs}

# Backward-compatible aliases (train's own mapping), kept only in case anything else
# still references these names directly; all fixed cells below use the _by_split dicts.
catid_to_idx    = catid_to_idx_by_split.get('train', {})
catid_to_contig = catid_to_contig_by_split.get('train', {})


### Step 3 — Dataset summary (real counts)
*Why:* images/annotations and average objects per image describe the task scale.


In [ ]:
rows = []
for split, c in coco.items():
    rows.append({'Split': split, 'Images': len(c.getImgIds()),
                 'Annotations': len(c.getAnnIds()),
                 'Avg ann/img': round(len(c.getAnnIds())/max(len(c.getImgIds()),1), 2)})
summary_df = pd.DataFrame(rows)
print('DATASET SUMMARY'); print('='*60)
print(summary_df.to_string(index=False))
print('='*60)
print(f"TOTAL: {summary_df['Images'].sum()} images, {summary_df['Annotations'].sum()} annotations")


### Step 4 — Verify the train / val / test split
*Why:* the proposal needs a proper split with **no image leakage**. CarDD ships pre-split; this proves the splits are disjoint and reports proportions.


In [ ]:
sets = {}
for split, c in coco.items():
    sets[split] = set(im['file_name'] for im in c.loadImgs(c.getImgIds()))
ov_tv = sets['train'] & sets['val']; ov_tt = sets['train'] & sets['test']; ov_vt = sets['val'] & sets['test']
total = sum(len(s) for s in sets.values())
print('SPLIT CHECK'); print('='*50)
for split, s in sets.items():
    print(f'   {split:<6}: {len(s):>5} images  ({100*len(s)/total:.1f}%)')
print('-'*50)
print('   train-val  overlap:', len(ov_tv))
print('   train-test overlap:', len(ov_tt))
print('   val-test   overlap:', len(ov_vt))
print('No leakage - splits are disjoint.' if not (ov_tv or ov_tt or ov_vt)
      else 'WARNING: overlapping images between splits!')


### Step 5 — Class distribution & imbalance (EDA)
*Why:* imbalance is central to the proposal; the ratio explains weak per-class results later and motivates augmentation/weighting.


In [ ]:
counts = defaultdict(lambda: defaultdict(int))
for split, c in coco.items():
    for ann in c.loadAnns(c.getAnnIds()):
        counts[split][CAT_ID_TO_NAME.get(ann['category_id'], '?')] += 1
dist_df = pd.DataFrame(counts).fillna(0).astype(int)
dist_df['TOTAL'] = dist_df.sum(axis=1)
dist_df = dist_df.sort_values('TOTAL', ascending=False)
print('CLASS DISTRIBUTION (instances)'); print('='*70)
print(dist_df.to_string())
if dist_df['TOTAL'].min() > 0:
    print('='*70)
    print(f"Imbalance ratio (max/min): {dist_df['TOTAL'].max()/dist_df['TOTAL'].min():.1f}x")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
t = dist_df['TOTAL']
ax[0].bar(t.index, t.values, color=sns.color_palette('viridis', len(t)), edgecolor='black')
ax[0].set_title('Instances per class', fontweight='bold'); ax[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(t.values):
    ax[0].text(i, v, str(int(v)), ha='center', va='bottom', fontweight='bold')
ax[1].pie(t.values, labels=t.index, autopct='%1.1f%%', startangle=90,
          colors=sns.color_palette('viridis', len(t)))
ax[1].set_title('Class proportion', fontweight='bold')
plt.tight_layout(); plt.savefig('class_distribution.png', dpi=200, bbox_inches='tight'); plt.show()
print('Saved: class_distribution.png')


### Step 6 — Image dimensions & bounding-box sizes (EDA)
*Why:* shows whether resizing to 640 is reasonable and which classes are small (small objects are harder - useful to explain later results).


In [ ]:
dims, bboxes = [], []
for split, c in coco.items():
    lut = {im['id']: im for im in c.loadImgs(c.getImgIds())}
    for im in lut.values():
        dims.append({'width': im.get('width'), 'height': im.get('height')})
    for ann in c.loadAnns(c.getAnnIds()):
        x, y, bw, bh = ann['bbox']
        im = lut.get(ann['image_id'], {}); iw, ih = im.get('width', 0), im.get('height', 0)
        bboxes.append({'class': CAT_ID_TO_NAME.get(ann['category_id'], '?'),
                       'rel_area': (bw*bh)/max(iw*ih, 1) if iw and ih else np.nan})
dim_df, bbox_df = pd.DataFrame(dims), pd.DataFrame(bboxes)
print('IMAGE DIMENSIONS'); print('='*50)
print(dim_df.describe().round(1).to_string())

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].scatter(dim_df['width'], dim_df['height'], alpha=0.3, s=10)
ax[0].set_title('Image width vs height'); ax[0].set_xlabel('width'); ax[0].set_ylabel('height')
order = bbox_df.groupby('class')['rel_area'].median().sort_values().index
sns.boxplot(data=bbox_df, x='class', y='rel_area', order=order, showfliers=False, ax=ax[1])
ax[1].set_title('Relative bbox area by class (smaller = harder)'); ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig('dimensions_and_bboxes.png', dpi=200, bbox_inches='tight'); plt.show()
print('Saved: dimensions_and_bboxes.png')


### Step 7 — Data integrity audit
*Why:* before cleaning you must *find* problems - missing files, unreadable images, zero-area boxes, out-of-bounds boxes.


In [ ]:
issues = {'missing': [], 'corrupt': [], 'zero_area': [], 'out_of_bounds': []}
for split, c in coco.items():
    lut = {im['id']: im for im in c.loadImgs(c.getImgIds())}
    for im in lut.values():
        fp = os.path.join(IMG_DIRS[split], im['file_name'])
        if not os.path.exists(fp):
            issues['missing'].append(fp); continue
        try:
            with Image.open(fp) as o: o.verify()
        except Exception:
            issues['corrupt'].append(fp)
    for ann in c.loadAnns(c.getAnnIds()):
        x, y, bw, bh = ann['bbox']
        im = lut.get(ann['image_id'], {}); iw, ih = im.get('width', 0), im.get('height', 0)
        if bw <= 0 or bh <= 0:
            issues['zero_area'].append(ann['id'])
        elif iw and ih and (x < 0 or y < 0 or x+bw > iw+1 or y+bh > ih+1):
            issues['out_of_bounds'].append(ann['id'])
print('DATA INTEGRITY REPORT'); print('='*50)
for k, v in issues.items():
    print(f'   {k:<15}: {len(v)}')
json.dump({k: len(v) for k, v in issues.items()}, open('integrity_report.json', 'w'), indent=2)
print('Saved: integrity_report.json')


### Step 8 — Clean the data
*Why:* produce a **cleaned dataset** (a proposal deliverable). Rules: drop zero-area boxes, clip out-of-bounds boxes, skip missing/corrupt images. We write cleaned COCO JSONs (keep originals intact) and use them downstream.


In [ ]:
def clean_split(split):
    ann_file = os.path.join(DATA_ROOT, 'annotations', SPLIT_MAP[split][0])
    if not os.path.exists(ann_file): return None
    data = json.load(open(ann_file))
    img_lut = {im['id']: im for im in data['images']}
    valid_imgs = {im['id'] for im in data['images']
                  if os.path.exists(os.path.join(IMG_DIRS[split], im['file_name']))}
    kept, dropped, clipped = [], 0, 0
    for a in data['annotations']:
        if a['image_id'] not in valid_imgs:
            dropped += 1; continue
        x, y, bw, bh = a['bbox']
        if bw <= 0 or bh <= 0:
            dropped += 1; continue
        im = img_lut.get(a['image_id']); iw, ih = im['width'], im['height']
        nx, ny = max(0.0, x), max(0.0, y)
        nx2, ny2 = min(x + bw, iw), min(y + bh, ih)
        nw, nh = nx2 - nx, ny2 - ny
        if nw <= 0 or nh <= 0:
            dropped += 1; continue
        if (nx, ny, nw, nh) != (x, y, bw, bh):
            clipped += 1
        a = {**a, 'bbox': [nx, ny, nw, nh], 'area': nw * nh}
        kept.append(a)
    data['annotations'] = kept
    out = os.path.join(CLEAN_ANN_DIR, SPLIT_MAP[split][0])
    json.dump(data, open(out, 'w'))
    return {'split': split, 'kept': len(kept), 'dropped': dropped, 'clipped': clipped}

print('CLEANING'); print('='*60)
clean_log = [r for r in (clean_split(s) for s in ['train', 'val', 'test']) if r]
print(pd.DataFrame(clean_log).to_string(index=False))
print('Cleaned annotation JSONs written to:', CLEAN_ANN_DIR)
ANN_DIR = CLEAN_ANN_DIR
coco = {s: COCO(os.path.join(ANN_DIR, SPLIT_MAP[s][0]))
        for s in SPLIT_MAP if os.path.exists(os.path.join(ANN_DIR, SPLIT_MAP[s][0]))}
print('Reloaded COCO from cleaned annotations.')


### Step 9 — Visualise annotated samples
*Why:* human sanity-check that labels line up with real damage before training.


In [ ]:
def show_samples(split='train', n=6):
    c = coco[split]; ids = c.getImgIds()
    pick = np.random.choice(ids, size=min(n, len(ids)), replace=False)
    pal = sns.color_palette('bright', len(CLASS_NAMES))
    cmap = {n_: pal[i] for i, n_ in enumerate(CLASS_NAMES)}
    rows = (len(pick)+2)//3
    fig, axes = plt.subplots(rows, 3, figsize=(18, 5*rows)); axes = np.array(axes).reshape(-1)
    for ax, iid in zip(axes, pick):
        im = c.loadImgs([iid])[0]
        try: img = Image.open(os.path.join(IMG_DIRS[split], im['file_name'])).convert('RGB')
        except Exception: ax.axis('off'); continue
        ax.imshow(img)
        for ann in c.loadAnns(c.getAnnIds(imgIds=[iid])):
            x, y, bw, bh = ann['bbox']; name = CAT_ID_TO_NAME.get(ann['category_id'], '?')
            col = cmap.get(name, 'red')
            ax.add_patch(patches.Rectangle((x, y), bw, bh, linewidth=2, edgecolor=col, facecolor='none'))
            ax.text(x, max(y-5, 0), name, color='white', fontsize=9,
                    bbox=dict(facecolor=col, alpha=0.7, pad=1))
        ax.set_title(im['file_name'], fontsize=9); ax.axis('off')
    for ax in axes[len(pick):]: ax.axis('off')
    plt.tight_layout(); plt.savefig(f'samples_{split}.png', dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: samples_{split}.png')

np.random.seed(42); show_samples('train', 6)


## Part B — Preprocessing: resize/normalise, split, augment, convert  (Objective 2.2)

### Step 10 — Resize & normalise (demonstration)
*Why:* the proposal lists "resize and normalise images". Each model expects different input:
- **YOLOv8**: resized to 640 and scaled to [0,1] internally.
- **Faster R-CNN / SSD / RetinaNet**: torchvision resizes + normalises (ImageNet mean/std) **inside the model**.
- **ResNet50**: explicit `Resize(224)` + ImageNet `Normalize` (Step 15).

This cell demonstrates the operation on one image so the effect is visible.


In [ ]:
IMGSZ = 640
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
sample_path = sorted(glob.glob(os.path.join(IMG_DIRS['train'], '*')))[0]
orig = Image.open(sample_path).convert('RGB')
resized = orig.resize((IMGSZ, IMGSZ))
arr = np.asarray(resized).astype(np.float32) / 255.0
norm = (arr - MEAN) / STD
print('original size:', orig.size, '-> resized:', resized.size)
print('range after /255 :', round(float(arr.min()),3), 'to', round(float(arr.max()),3))
print('range after norm :', round(float(norm.min()),3), 'to', round(float(norm.max()),3))
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(orig); ax[0].set_title(f'Original {orig.size}'); ax[0].axis('off')
ax[1].imshow(arr);  ax[1].set_title(f'Resized {IMGSZ}x{IMGSZ}, /255'); ax[1].axis('off')
ax[2].imshow((norm - norm.min())/(norm.max()-norm.min())); ax[2].set_title('Normalised (display-scaled)'); ax[2].axis('off')
plt.tight_layout(); plt.savefig('preprocess_resize_normalise.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: preprocess_resize_normalise.png')


### Step 11 — Data augmentation (demonstration): horizontal flip + brightness
*Why:* the proposal names these two specifically to improve generalisation and offset imbalance. Shown here; applied automatically in training (torchvision dataset Step 14; YOLO via `fliplr`/`hsv_v`).


In [ ]:
img = Image.open(sample_path).convert('RGB').resize((IMGSZ, IMGSZ))
flip   = img.transpose(Image.FLIP_LEFT_RIGHT)
bright = ImageEnhance.Brightness(img).enhance(1.4)
dark   = ImageEnhance.Brightness(img).enhance(0.6)
fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for a, im_, ttl in zip(ax, [img, flip, bright, dark],
                       ['Original', 'Horizontal flip', 'Brightness +40%', 'Brightness -40%']):
    a.imshow(im_); a.set_title(ttl); a.axis('off')
plt.tight_layout(); plt.savefig('augmentation_demo.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: augmentation_demo.png')


### Step 12 — Class weights for imbalance
*Why:* inverse-frequency weights quantify how under-represented each class is.


In [ ]:
counts_total = {n: int(dist_df.loc[n, 'TOTAL']) for n in dist_df.index}
tot, k = sum(counts_total.values()), len(counts_total)
class_weights = {n: round(tot/(k*c), 3) if c else 0.0 for n, c in counts_total.items()}
print('CLASS WEIGHTS (inverse frequency)'); print('='*50)
for n, w in sorted(class_weights.items(), key=lambda x: -x[1]):
    print(f'   {n:<16} count={counts_total[n]:>6}  weight={w}')
json.dump(class_weights, open('class_weights.json', 'w'), indent=2)
print('Saved: class_weights.json')


### Step 13 — Convert annotations to YOLO format
*Why:* YOLOv8 needs `images/`, `labels/` and normalised `cx cy w h`. Uses cleaned annotations from Step 8.


In [ ]:
def convert_split(split):
    ann, sub = SPLIT_MAP[split]
    p = os.path.join(ANN_DIR, ann)
    if not os.path.exists(p): print(f'  skip {split}'); return
    c = COCO(p)
    out_img = os.path.join(YOLO_ROOT, 'images', split)
    out_lbl = os.path.join(YOLO_ROOT, 'labels', split)
    os.makedirs(out_img, exist_ok=True); os.makedirs(out_lbl, exist_ok=True)
    n = 0
    for im in c.loadImgs(c.getImgIds()):
        iw, ih, fname = im['width'], im['height'], im['file_name']
        src = os.path.join(IMG_DIRS[split], fname)
        if not os.path.exists(src): continue
        dst = os.path.join(out_img, fname)
        if not os.path.exists(dst):
            try: os.symlink(os.path.abspath(src), dst)
            except Exception: shutil.copy(src, dst)
        lines = []
        for ann_ in c.loadAnns(c.getAnnIds(imgIds=[im['id']])):
            x, y, bw, bh = ann_['bbox']
            if bw <= 0 or bh <= 0: continue
            cx, cy, nw, nh = (x+bw/2)/iw, (y+bh/2)/ih, bw/iw, bh/ih
            cx, cy, nw, nh = [min(max(v, 0.0), 1.0) for v in (cx, cy, nw, nh)]
            lines.append(f"{catid_to_idx_by_split[split][ann_['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        open(os.path.join(out_lbl, os.path.splitext(fname)[0]+'.txt'), 'w').write('\n'.join(lines))
        n += 1
    print(f'  {split}: {n} images converted')

print('Converting cleaned COCO -> YOLO...')
for s in ['train', 'val', 'test']: convert_split(s)

import yaml
data_yaml = {'path': os.path.abspath(YOLO_ROOT), 'train': 'images/train',
             'val': 'images/val', 'test': 'images/test',
             'names': {i: n for i, n in enumerate(CLASS_NAMES)}}
yaml_path = os.path.join(YOLO_ROOT, 'data.yaml')
yaml.safe_dump(data_yaml, open(yaml_path, 'w'), sort_keys=False)
print('Wrote', yaml_path)


### Step 14 — Build the torchvision dataset (resize/normalise + flip + brightness baked in)
*Why:* Faster R-CNN, SSD and RetinaNet read straight from cleaned COCO via one `Dataset`. Flip + brightness applied here so all three train on identical, fairly-augmented data. Torchvision detectors normalise internally, so we only convert to tensor + augment.


In [ ]:
import torchvision
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader, Dataset

class CocoDetDataset(Dataset):
    '''Reads cleaned CarDD COCO. Returns (image_tensor, target, path). Labels 1-based (0=background).
    Images with zero boxes surviving clipping are excluded entirely (no dummy/background boxes —
    a synthesized box would contradict the 1-based/background-is-0 convention used by the
    torchvision detectors and would corrupt evaluation once converted back to 0-based labels).'''
    def __init__(self, split, train_aug=False):
        self.split = split
        self.coco = COCO(os.path.join(ANN_DIR, SPLIT_MAP[split][0]))
        self.img_dir = IMG_DIRS[split]
        candidates = [i for i in sorted(self.coco.getImgIds())
                      if len(self.coco.getAnnIds(imgIds=[i])) > 0]
        self.ids = [i for i in candidates if len(self._clipped_boxes(i)[0]) > 0]
        dropped = len(candidates) - len(self.ids)
        if dropped:
            print(f'  [{split}] dropped {dropped} image(s) with no boxes surviving clipping')
        self.train_aug = train_aug
    def _clipped_boxes(self, iid):
        info = self.coco.loadImgs([iid])[0]; iw, ih = info['width'], info['height']
        boxes, labels = [], []
        for a in self.coco.loadAnns(self.coco.getAnnIds(imgIds=[iid])):
            x, y, bw, bh = a['bbox']
            x1, y1 = max(x, 0), max(y, 0); x2, y2 = min(x+bw, iw), min(y+bh, ih)
            if x2 <= x1 or y2 <= y1: continue
            boxes.append([x1, y1, x2, y2]); labels.append(catid_to_contig_by_split[self.split][a['category_id']])
        return boxes, labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        iid = self.ids[idx]; info = self.coco.loadImgs([iid])[0]
        path = os.path.join(self.img_dir, info['file_name'])
        img = Image.open(path).convert('RGB')
        if self.train_aug and torch.rand(1).item() < 0.5:
            factor = float(torch.empty(1).uniform_(0.6, 1.4))
            img = ImageEnhance.Brightness(img).enhance(factor)
        boxes, labels = self._clipped_boxes(iid)
        # self.ids is pre-filtered to images with >=1 box, so boxes is never empty here.
        img = F.to_tensor(img)
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        if self.train_aug and torch.rand(1).item() < 0.5:
            img = torch.flip(img, dims=[2]); w = img.shape[2]
            boxes = boxes.clone(); boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
        return img, {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([iid])}, path

def collate(batch): return tuple(zip(*batch))
print('torchvision dataset ready. train images:', len(CocoDetDataset('train')))


### Step 15 — ResNet50 classifier transforms (explicit resize + normalise)
*Why:* the classification baseline uses the textbook 224x224 + ImageNet normalisation; flip + brightness folded into training.


In [ ]:
import torchvision.transforms as T
train_tf = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(),
                      T.ColorJitter(brightness=0.4),
                      T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
eval_tf  = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
print('Classifier transforms ready (resize 224, flip, brightness, normalise).')


## Part C — Train the six models  (Objectives 2.3, 2.4)

Five CNN baselines (YOLOv8, Faster R-CNN, SSD, EfficientDet, ResNet50) plus one hybrid CNN-Transformer detector (RT-DETR, C6) so the comparison spans the transformer-based paradigm as well as one-stage / two-stage CNNs.

Set every `EPOCHS_*` low for a first run, then raise for final numbers.


In [ ]:
# Training epochs — FULL (final-numbers) budget. Total is roughly 5-7 h on a T4 (less on a
# P100), which FITS in a single Kaggle session (~9-12 h) — so the recommended path is one
# committed background run: Save Version -> Save & Run All. The crash-proofing is still
# active: every training cell SKIPs (already finished) or RESUMEs (interrupted) from
# checkpoints under CKPT_ROOT, so an interactive session that dies can be re-run and it
# fast-forwards past completed models and continues the interrupted one.
# (For a quick pipeline check instead, drop these to e.g. 12, 6, 3 and C6's EPOCHS_RTDETR to 12.)
EPOCHS_YOLO, EPOCHS_TV, EPOCHS_CLS = 30, 10, 6
RETRAIN_IF_CHECKPOINT = False   # True = ignore finished checkpoints and retrain everything from scratch
BATCH_YOLO  = 16
BATCH_TV    = 4          # with AMP this fits easily on a 16GB T4 at TV_MIN_SIZE=512; try 8 if headroom.
LR_TV       = 0.005      # Faster R-CNN / RetinaNet at batch 4 (reference: 0.02 @ batch 16 -> linear-scaled).
LR_EFFDET   = 5e-4       # EfficientDet-D0 (real effdet path in C4). Focal loss on a fresh head
                         # also dislikes LR_TV=0.005 at batch 4; C4 now trains at this LR with a
                         # NaN guard + gradient clipping (same protections as SSD).
LR_SSD      = 5e-4       # SSD300-VGG16 ONLY. torchvision's SSD recipe is lr 0.002 at an EFFECTIVE
                         # batch of 32 (8 GPUs x batch 4) -> ~0.00025 at our batch 4. Reusing LR_TV
                         # (0.005, ~20x hotter) on a VGG16 backbone with no BatchNorm makes the SSD
                         # multibox loss explode to inf/NaN in epoch 1 -> C3 dies, ssd.pt is never
                         # written, and Part D later fails with 'ssd.pt not found - re-run C3'.
TV_MIN_SIZE = 512        # input short-side for Faster R-CNN / RetinaNet (default 800). Smaller = far less memory.
TV_MAX_SIZE = 800        # input long-side cap (default 1333).
TV_WORKERS  = 2          # DataLoader workers (Kaggle GPU VMs have 4 vCPUs; 2 is a safe setting).
SEED        = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if DEVICE == 'cpu':
    print('WARNING: no GPU detected. Training the torchvision detectors (Faster R-CNN/SSD/RetinaNet)\n'
          '         on CPU will likely exhaust system RAM and crash the session.\n'
          '         On Colab: Runtime -> Change runtime type -> T4 GPU (then Run all),\n'
          '         then re-run the notebook from the top.')
print(f'YOLO {EPOCHS_YOLO} ep | torchvision {EPOCHS_TV} ep | classifier {EPOCHS_CLS} ep | device {DEVICE}')

def ultralytics_run_finished(pt):
    '''True if an Ultralytics checkpoint belongs to a COMPLETED run (it writes epoch=-1 on finish).'''
    try:
        return torch.load(pt, map_location='cpu', weights_only=False).get('epoch', 0) == -1
    except Exception:
        return False


### C1 — YOLOv8 (one-stage CNN, primary baseline). `fliplr` + `hsv_v` = flip/brightness augmentation.

In [ ]:
# Self-heal: guarantee ultralytics is present even if cell 2 (or
# ensure_packages) never ran in this runtime (e.g. after a Colab
# disconnect/reconnect mid-session). Fully standalone - does not
# depend on any earlier cell having executed.
import importlib as _importlib, subprocess as _subprocess, sys as _sys
try:
    _importlib.import_module('ultralytics')
except ImportError:
    print('ultralytics missing in this runtime - installing ultralytics now...')
    _subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    print('Done.')

from ultralytics import YOLO
# Train into CKPT_ROOT (Drive-backed) instead of a local relative path, so best.pt
# survives a full Colab runtime recycle, not just a kernel restart.
_yolo_project = globals().get('CKPT_ROOT', 'runs_cardd')
_yolo_w = os.path.join(_yolo_project, 'yolov8s', 'weights')
_yolo_last, _yolo_best = os.path.join(_yolo_w, 'last.pt'), os.path.join(_yolo_w, 'best.pt')

# Crash-proofing: SKIP if this run already finished, RESUME if it was interrupted mid-training,
# otherwise train fresh. This is what lets Run All fast-forward after a Colab disconnect.
if (os.path.exists(_yolo_best) and os.path.exists(_yolo_last)
        and ultralytics_run_finished(_yolo_last) and not RETRAIN_IF_CHECKPOINT):
    print('YOLOv8: FINISHED run found on Drive -> skipping training, using', _yolo_best)
elif os.path.exists(_yolo_last) and not RETRAIN_IF_CHECKPOINT:
    print('YOLOv8: INTERRUPTED run found -> resuming from', _yolo_last)
    yolo = YOLO(_yolo_last)
    try:
        yolo.train(resume=True)
    except Exception as _e:
        if os.path.exists(_yolo_best):
            print('YOLOv8: resume not needed/possible ->', _e, '\n  Using existing', _yolo_best)
        else:
            raise
else:
    yolo = YOLO('yolov8s.pt')
    yolo.train(data=yaml_path, epochs=EPOCHS_YOLO, imgsz=IMGSZ, batch=BATCH_YOLO,
               device=(0 if DEVICE=='cuda' else 'cpu'), project=_yolo_project, name='yolov8s',
               exist_ok=True, seed=SEED, fliplr=0.5, hsv_v=0.4)
# Ultralytics may save under its own runs dir (e.g. runs/detect/...), so read the real path
# from the trainer rather than assuming it. Fall back to a recursive search if needed.
YOLO_BEST = None
try:
    YOLO_BEST = str(yolo.trainer.best)            # exact path to best.pt for this run
except Exception:
    pass
if not (YOLO_BEST and os.path.exists(YOLO_BEST)):
    alt = (sorted(glob.glob(os.path.join(_yolo_project, 'yolov8s*/weights/best.pt'))) or
           sorted(glob.glob('**/weights/best.pt', recursive=True)) or
           sorted(glob.glob('/content/drive/MyDrive/cardd_ckpts/**/weights/best.pt', recursive=True) + glob.glob('/content/**/weights/best.pt', recursive=True)) or
           sorted(glob.glob('**/weights/last.pt', recursive=True)))
    YOLO_BEST = alt[-1] if alt else YOLO_BEST
assert YOLO_BEST and os.path.exists(YOLO_BEST), (
    'YOLO weights not found — C1 training did not complete in this session. '
    'Re-run this YOLOv8 cell to the end (or Kernel > Restart & Run All after a crash) '
    'before running evaluation.')
print('YOLOv8 done ->', YOLO_BEST)
# Free the GPU: best.pt is on disk (Drive), and evaluation reloads from YOLO_BEST,
# so keeping the trained model object resident just eats VRAM for the rest of the session.
if 'yolo' in globals(): del yolo
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


### Shared torchvision machinery — one model factory + one training loop (used by C2-C4)

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

def build_detector(name, num_classes):
    '''num_classes INCLUDES background. Fresh head for our classes.'''
    if name == 'fasterrcnn':
        m = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
        inf = m.roi_heads.box_predictor.cls_score.in_features
        m.roi_heads.box_predictor = FastRCNNPredictor(inf, num_classes)
        m.transform.min_size = (TV_MIN_SIZE,); m.transform.max_size = TV_MAX_SIZE  # lower memory
    elif name == 'ssd':
        m = torchvision.models.detection.ssd300_vgg16(weights='DEFAULT')
        in_ch = [c.in_channels for c in m.head.classification_head.module_list]
        n_anchors = m.anchor_generator.num_anchors_per_location()
        m.head.classification_head = SSDClassificationHead(in_ch, n_anchors, num_classes)
    elif name == 'retinanet':
        m = torchvision.models.detection.retinanet_resnet50_fpn(weights='DEFAULT')
        n_anchors = m.head.classification_head.num_anchors
        in_ch = m.backbone.out_channels
        m.head.classification_head = RetinaNetClassificationHead(in_ch, n_anchors, num_classes)
        m.transform.min_size = (TV_MIN_SIZE,); m.transform.max_size = TV_MAX_SIZE  # lower memory
    else:
        raise ValueError(name)
    return m

def _make_train_loader(batch):
    # pin_memory: page-locked host memory -> faster (and async) CPU->GPU copies.
    # persistent_workers: keep workers alive between epochs instead of re-forking every epoch.
    # prefetch_factor: each worker keeps a queue of decoded batches ready so the GPU never waits.
    kw = dict(batch_size=batch, shuffle=True, collate_fn=collate,
              num_workers=TV_WORKERS, pin_memory=(DEVICE == 'cuda'))
    if TV_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=2)  # 2 keeps Colab system RAM safe with large CarDD images
    return DataLoader(CocoDetDataset('train', train_aug=True), **kw)

# ---- Per-model training hyper-parameters -------------------------------------------
# LR_TV (0.005) is the linearly-scaled torchvision reference LR for Faster R-CNN /
# RetinaNet at batch 4. SSD300-VGG16 is different: its reference recipe is lr 0.002 at
# an EFFECTIVE batch of 32 (8 GPUs x batch 4), i.e. ~0.00025 at batch 4 - and its VGG16
# backbone has no BatchNorm to absorb an over-hot step. Training SSD at 0.005 makes the
# multibox loss blow up to inf/NaN within the first epoch, so the model "never runs".
# SSD therefore gets: a much lower LR, AMP off (plain VGG16 is fp16-fragile), a linear
# LR warmup, and gradient-norm clipping. The other two keep their proven settings.
def _tv_hp(name):
    if name == 'ssd':
        return dict(lr=globals().get('LR_SSD', 5e-4), amp=False, clip=10.0)
    return dict(lr=globals().get('LR_TV', 0.005), amp=True, clip=None)

def _params_finite(model):
    """False if any weight is NaN/Inf (i.e. the checkpoint came from a diverged run)."""
    return all(torch.isfinite(p).all() for p in model.parameters())

def train_detector(name, epochs, batch=None, lr=None, amp=None, clip='auto'):
    # Resolve at call-time (not as default-arg values) so this function defines cleanly
    # even if the Part C config cell (BATCH_TV/LR_TV/LR_SSD) has not been run yet.
    hp = _tv_hp(name)
    batch = BATCH_TV if batch is None else batch
    lr    = hp['lr']   if lr   is None    else lr
    amp   = hp['amp']  if amp  is None    else amp
    clip  = hp['clip'] if clip == 'auto'  else clip
    _ckpt_dir = os.path.join(globals().get('CKPT_ROOT', 'runs_cardd'), 'torchvision')
    os.makedirs(_ckpt_dir, exist_ok=True)
    ckpt = os.path.join(_ckpt_dir, f'{name}.pt')            # final weights (written once, at the end)
    resume_pt = os.path.join(_ckpt_dir, f'{name}_resume.pt') # rolling per-epoch state (deleted on success)
    # SKIP: a final checkpoint with no leftover resume file means this model finished in a
    # previous session. Load it and return instantly instead of burning hours retraining.
    # NEW: verify the weights are finite first - a run that diverged to NaN can still have
    # saved a (garbage) ssd.pt, and silently reusing it would poison every Part D number.
    if os.path.exists(ckpt) and not os.path.exists(resume_pt) and not globals().get('RETRAIN_IF_CHECKPOINT', False):
        model = build_detector(name, NUM_CLASSES + 1)
        model.load_state_dict(torch.load(ckpt, map_location='cpu'))
        if _params_finite(model):
            print(f'\n=== {name}: FINISHED checkpoint found ({ckpt}) -> skipping training ===')
            return model.cpu().eval()
        print(f'\n!! {ckpt} contains NaN/Inf weights (saved by a previous DIVERGED run).')
        print(f'!! Ignoring it and retraining {name} from scratch with the fixed settings.')
        del model
    use_amp = bool(amp) and (DEVICE == 'cuda')
    print(f'\n=== Training {name} ({epochs} epochs, batch {batch}, lr {lr:g}, '
          f'AMP={"on" if use_amp else "off"}, grad-clip={clip}) ===')
    train_ld = _make_train_loader(batch)
    model = build_detector(name, NUM_CLASSES + 1).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs//3), gamma=0.1)
    # Automatic Mixed Precision: forward/backward mostly in fp16 -> ~2x faster + ~half the
    # memory on a T4, with GradScaler guarding against fp16 underflow. Off for SSD (VGG16).
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)   # torch >= 2.3 API
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)      # older torch fallback
    # RESUME: pick up exactly where an interrupted run stopped (model + optimizer + scheduler
    # + AMP scaler state are all restored). NEW: if the saved state is NaN/Inf (the old,
    # diverged SSD run left one behind on Drive), throw it away and start fresh instead.
    start_ep = 0
    if os.path.exists(resume_pt) and not globals().get('RETRAIN_IF_CHECKPOINT', False):
        _st = torch.load(resume_pt, map_location=DEVICE)
        model.load_state_dict(_st['model'])
        if _params_finite(model):
            opt.load_state_dict(_st['opt']); sched.load_state_dict(_st['sched'])
            try:
                scaler.load_state_dict(_st['scaler'])
            except Exception:
                pass   # e.g. resume file written with AMP on, now running with AMP off
            start_ep = _st['epoch']
            print(f'  resuming {name} from epoch {start_ep+1}/{epochs} (found {resume_pt})')
        else:
            print(f'  !! {resume_pt} holds NaN/Inf weights from a diverged run -> deleting it and starting {name} FRESH.')
            os.remove(resume_pt)
            model = build_detector(name, NUM_CLASSES + 1).to(DEVICE)
            params = [p for p in model.parameters() if p.requires_grad]
            opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
            sched = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs//3), gamma=0.1)
    n_batches = len(train_ld)
    log_every = max(1, n_batches // 5)
    # Linear LR warmup over the first ~500 iterations of a FRESH run (torchvision's own
    # detection recipes do the same). Ramping 10% -> 100% of lr protects the freshly
    # re-initialised classification head from kicking the pretrained backbone into NaN.
    warmup_total = min(500, max(n_batches - 1, 1)) if start_ep == 0 else 0
    bad_streak = 0
    for ep in range(start_ep, epochs):
        model.train(); t0 = time.time()
        running = torch.zeros((), device=DEVICE); nb = 0   # accumulate ON-device: no per-step GPU sync
        for bi, (imgs, targets, _) in enumerate(train_ld, 1):
            if warmup_total and ep == 0:
                if bi <= warmup_total:
                    f = 0.1 + 0.9 * (bi / warmup_total)
                    for g in opt.param_groups: g['lr'] = lr * f
                elif bi == warmup_total + 1:
                    for g in opt.param_groups: g['lr'] = lr
            imgs = [i.to(DEVICE, non_blocking=True) for i in imgs]
            targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items() if k != 'image_id'}
                       for t in targets]
            with torch.autocast(device_type='cuda', enabled=use_amp):
                loss = sum(model(imgs, targets).values())
            # NaN guard: a non-finite loss means this step would destroy the weights.
            # Skip the batch; if it keeps happening, stop LOUDLY with the actual fix,
            # instead of silently "finishing" and saving a garbage checkpoint.
            if not torch.isfinite(loss):
                bad_streak += 1
                opt.zero_grad(set_to_none=True)
                if bad_streak in (1, 5, 10, 20):
                    print(f'    !! epoch {ep+1} batch {bi}: non-finite loss - skipping (streak {bad_streak})')
                if bad_streak >= 25:
                    raise RuntimeError(
                        f'{name}: training DIVERGED (25 consecutive non-finite losses). '
                        f'The learning rate is too high for this model - re-run this cell with e.g. '
                        f'train_detector("{name}", EPOCHS_TV, lr={lr/5:g}).')
                continue
            bad_streak = 0
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if clip:
                scaler.unscale_(opt)   # no-op when AMP is off
                torch.nn.utils.clip_grad_norm_(params, max_norm=clip)
            scaler.step(opt); scaler.update()
            running += loss.detach(); nb += 1
            if bi % log_every == 0:   # heartbeat so long epochs never look hung
                print(f'    epoch {ep+1}: batch {bi}/{n_batches}  '
                      f'({bi/(time.time()-t0):.2f} it/s)', flush=True)
        sched.step()
        print(f'  epoch {ep+1}/{epochs}  loss={float(running)/max(nb,1):.4f}  '
              f'({time.time()-t0:.0f}s)')
        # rolling resume checkpoint on Drive: a crash now costs at most one epoch of work
        torch.save({'epoch': ep + 1, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'sched': sched.state_dict(), 'scaler': scaler.state_dict()}, resume_pt)
    torch.save(model.state_dict(), ckpt); print('  saved', ckpt)
    if os.path.exists(resume_pt): os.remove(resume_pt)   # finished cleanly -> resume state obsolete
    # --- Memory policy: only ONE model lives on the GPU at a time. ---
    # Park the finished model on CPU (Part D's predict_* functions move it back to the
    # GPU just for inference). Also drop the optimizer/scaler (they hold momentum buffers
    # as big as the model) and the DataLoader (shuts down its persistent workers -> frees
    # system RAM). Without this, running C2+C3+C4 stacks three detectors on the GPU and
    # Colab runs out of memory.
    model = model.cpu().eval()
    del opt, sched, scaler, train_ld
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f'  {name} parked on CPU; GPU memory released.')
    return model


### C2 — Faster R-CNN (two-stage, ResNet50-FPN backbone)

In [ ]:
frcnn = train_detector('fasterrcnn', EPOCHS_TV)

### C3 — SSD (one-stage)

**Fixed:** SSD now trains with its **own** hyper-parameters instead of borrowing Faster R-CNN's. torchvision's reference recipe for SSD300-VGG16 is lr **0.002 at an effective batch of 32** (8 GPUs x 4), which linearly scales to ~**0.00025 at our batch of 4**. The old shared `LR_TV = 0.005` was ~20x hotter than that, and SSD's VGG16 backbone has **no BatchNorm** to absorb it — so the multibox loss exploded to inf/NaN in the first epoch, the cell died, `ssd.pt` was never written, and Part D then failed with *"ssd.pt not found - re-run the SSD training cell (C3) first"*.

The shared trainer now gives SSD: `lr = LR_SSD` (5e-4), **AMP off** (plain VGG16 is fp16-fragile), a **linear LR warmup** over the first ~500 iterations, **gradient-norm clipping** (max-norm 10), a **NaN guard** that stops loudly with the fix instead of saving garbage, and it **refuses to reuse a NaN checkpoint/resume file left behind by the old diverged run**. *Viva justification:* one-stage anchor losses on a BN-free backbone need a smaller, warmed-up step size; two-stage FPN models tolerate the larger scaled LR.


In [ ]:
# SSD trains with its OWN settings (applied automatically by train_detector via _tv_hp):
#   lr = LR_SSD (5e-4, NOT LR_TV)  |  AMP off  |  ~500-iter linear warmup  |  grad-clip 10
# With the old shared LR_TV=0.005 the SSD multibox loss went inf -> NaN inside epoch 1,
# so C3 never finished and Part D failed with "ssd.pt not found - re-run C3".
# NOTE: if an old, diverged ssd.pt / ssd_resume.pt is still on your Drive, the trainer
# now detects the NaN weights, ignores the file, and retrains fresh - no manual cleanup.
# If you EVER see repeated "non-finite loss" skips below, halve the LR and re-run, e.g.:
#   ssd = train_detector('ssd', EPOCHS_TV, lr=2.5e-4)
ssd = train_detector('ssd', EPOCHS_TV)


### C4 — EfficientDet-D0 (one-stage, BiFPN). Real `effdet` implementation, RetinaNet fallback.
EfficientDet is the most fragile of the five. Set `USE_EFFICIENTDET=True` (default) to train a **real EfficientDet-D0** via the `effdet` library with COCO-pretrained backbone+BiFPN and a fresh class head (transfer learning, per Objective 2.3/2.4). It uses its own fixed-size (512x512) letterboxed pipeline and anchor-based focal loss, so it does **not** share `train_detector`. If `effdet` is unavailable or errors, the cell automatically falls back to a torchvision **RetinaNet** one-stage stand-in. **State clearly in your write-up which one actually ran** (the printout and `EFFDET_NAME` tell you).

In [ ]:
# C4 — EfficientDet-D0 (real, via the `effdet` library) with automatic RetinaNet fallback.
# EfficientDet uses a fixed square input, ImageNet normalisation, yxyx boxes and 1-based
# classes, and anchor+focal loss — none of which match train_detector's torchvision API,
# so it gets its own dataset, training loop and predictor here.
USE_EFFICIENTDET = True
EFFDET_IMG_SIZE  = 512                       # EfficientDet-D0 native input
_IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _letterbox(pil_img, size=EFFDET_IMG_SIZE):
    """Resize (aspect-preserved) onto a size x size square. Returns tensor + transform meta."""
    W, H = pil_img.size
    s = size / max(W, H)
    nw, nh = round(W * s), round(H * s)
    pad_l, pad_t = (size - nw) // 2, (size - nh) // 2
    canvas = Image.new('RGB', (size, size), (124, 116, 104))   # ~ImageNet mean, padded region
    canvas.paste(pil_img.resize((nw, nh)), (pad_l, pad_t))
    t = F.to_tensor(canvas)
    t = (t - _IMNET_MEAN) / _IMNET_STD
    return t, s, pad_l, pad_t, W, H

def _unletterbox(boxes_xyxy, s, pad_l, pad_t, W, H):
    """Map xyxy boxes from model (square) space back to original image coords, clipped."""
    b = boxes_xyxy.copy().astype(np.float32)
    b[:, [0, 2]] = (b[:, [0, 2]] - pad_l) / s
    b[:, [1, 3]] = (b[:, [1, 3]] - pad_t) / s
    b[:, [0, 2]] = np.clip(b[:, [0, 2]], 0, W)
    b[:, [1, 3]] = np.clip(b[:, [1, 3]], 0, H)
    return b

class EffDetDataset(Dataset):
    """Letterboxed CarDD for EfficientDet. Target uses yxyx boxes in 512-space, 1-based cls.
    Images with no box surviving clipping are excluded (same rule as CocoDetDataset)."""
    MAX_DET = 100
    def __init__(self, split, train_aug=False):
        self.split = split
        self.coco = COCO(os.path.join(ANN_DIR, SPLIT_MAP[split][0]))
        self.img_dir = IMG_DIRS[split]
        cand = [i for i in sorted(self.coco.getImgIds())
                if len(self.coco.getAnnIds(imgIds=[i])) > 0]
        self.ids = [i for i in cand if len(self._boxes(i)[0]) > 0]
        self.train_aug = train_aug
    def _boxes(self, iid):
        info = self.coco.loadImgs([iid])[0]; iw, ih = info['width'], info['height']
        boxes, labels = [], []
        for a in self.coco.loadAnns(self.coco.getAnnIds(imgIds=[iid])):
            x, y, bw, bh = a['bbox']
            x1, y1 = max(x, 0), max(y, 0); x2, y2 = min(x + bw, iw), min(y + bh, ih)
            if x2 <= x1 or y2 <= y1: continue
            boxes.append([x1, y1, x2, y2])
            # split-aware mapping (same fix as CocoDetDataset): each split's COCO file may
            # number its category ids differently, so never reuse train's table blindly.
            labels.append(catid_to_contig_by_split[self.split][a['category_id']])
        return boxes, labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        iid = self.ids[idx]; info = self.coco.loadImgs([iid])[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        if self.train_aug and torch.rand(1).item() < 0.5:
            img = ImageEnhance.Brightness(img).enhance(float(torch.empty(1).uniform_(0.6, 1.4)))
        boxes, labels = self._boxes(iid)
        boxes = np.array(boxes, dtype=np.float32)
        if self.train_aug and torch.rand(1).item() < 0.5:        # horizontal flip
            img = img.transpose(Image.FLIP_LEFT_RIGHT); Wf = img.size[0]
            boxes[:, [0, 2]] = Wf - boxes[:, [2, 0]]
        t, s, pad_l, pad_t, W, H = _letterbox(img)
        # to model space, then xyxy -> yxyx for effdet
        boxes[:, [0, 2]] = boxes[:, [0, 2]] * s + pad_l
        boxes[:, [1, 3]] = boxes[:, [1, 3]] * s + pad_t
        yxyx = boxes[:, [1, 0, 3, 2]]
        cls = np.array(labels, dtype=np.int64)
        return t, torch.from_numpy(yxyx), torch.from_numpy(cls)

def effdet_collate(batch):
    imgs = torch.stack([b[0] for b in batch])
    B, M = len(batch), EffDetDataset.MAX_DET
    bbox = torch.zeros((B, M, 4), dtype=torch.float32)     # padding rows = zero box, cls 0
    clss = torch.zeros((B, M), dtype=torch.int64)          # (ignored by labeler: zero-area boxes never match)
    for i, (_, yxyx, cls) in enumerate(batch):
        n = min(len(cls), M)
        bbox[i, :n] = yxyx[:n]; clss[i, :n] = cls[:n]
    return imgs, {'bbox': bbox, 'cls': clss}

effdet_model = None            # DetBenchTrain (real) or torchvision RetinaNet (fallback)
effdet_predict_bench = None    # DetBenchPredict (real path only)
effdet_is_real = False
EFFDET_NAME = 'RetinaNet (EfficientDet stand-in)'

if USE_EFFICIENTDET:
    try:
        from effdet import create_model, DetBenchPredict
        # Crash-proofing: SKIP training entirely if a finished checkpoint is already on Drive.
        _ckpt_dir = os.path.join(globals().get('CKPT_ROOT', 'runs_cardd'), 'torchvision')
        _effdet_ckpt = os.path.join(_ckpt_dir, 'efficientdet_d0.pt')
        if os.path.exists(_effdet_ckpt) and not globals().get('RETRAIN_IF_CHECKPOINT', False):
            print('EfficientDet-D0: FINISHED checkpoint found -> skipping training,', _effdet_ckpt)
            bench = create_model('tf_efficientdet_d0', bench_task='train', num_classes=NUM_CLASSES,
                                 pretrained=False, pretrained_backbone=False, bench_labeler=True,
                                 image_size=(EFFDET_IMG_SIZE, EFFDET_IMG_SIZE))
            bench.model.load_state_dict(torch.load(_effdet_ckpt, map_location='cpu'))
            bench = bench.cpu().eval()
            effdet_model = bench
            effdet_predict_bench = DetBenchPredict(bench.model).eval()
        else:
            print('=== Training EfficientDet-D0 (effdet) ===')
            bench = create_model('tf_efficientdet_d0', bench_task='train', num_classes=NUM_CLASSES,
                                 pretrained=True, bench_labeler=True,
                                 image_size=(EFFDET_IMG_SIZE, EFFDET_IMG_SIZE)).to(DEVICE)
            _ld_kw = dict(batch_size=BATCH_TV, shuffle=True, collate_fn=effdet_collate,
                          num_workers=TV_WORKERS, pin_memory=(DEVICE == 'cuda'))
            if TV_WORKERS > 0:
                _ld_kw.update(persistent_workers=True, prefetch_factor=2)
            ld = DataLoader(EffDetDataset('train', train_aug=True), **_ld_kw)
            params = [p for p in bench.parameters() if p.requires_grad]
            # LR_EFFDET (not LR_TV): focal loss + fresh class head diverges at 0.005/batch-4,
            # the same failure mode the SSD cell had. Guarded below by a NaN skip + clip.
            opt = torch.optim.SGD(params, lr=globals().get('LR_EFFDET', 5e-4), momentum=0.9, weight_decay=5e-4)
            sched = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, EPOCHS_TV // 3), gamma=0.1)
            _use_amp = (DEVICE == 'cuda')
            try:
                _scaler = torch.amp.GradScaler('cuda', enabled=_use_amp)   # torch >= 2.3 API
            except (AttributeError, TypeError):
                _scaler = torch.cuda.amp.GradScaler(enabled=_use_amp)      # older torch fallback
            _bad = 0
            for ep in range(EPOCHS_TV):
                bench.train(); running, nb = torch.zeros((), device=DEVICE), 0
                _t0 = time.time()
                for imgs, target in ld:
                    imgs = imgs.to(DEVICE, non_blocking=True)
                    target = {'bbox': target['bbox'].to(DEVICE, non_blocking=True),
                              'cls': target['cls'].to(DEVICE, non_blocking=True)}
                    with torch.autocast(device_type='cuda', enabled=_use_amp):
                        loss = bench(imgs, target)['loss']
                    if not torch.isfinite(loss):     # NaN guard: never let a bad step poison the weights
                        _bad += 1
                        opt.zero_grad(set_to_none=True)
                        if _bad >= 25:
                            raise RuntimeError('EfficientDet: training DIVERGED (25 consecutive non-finite '
                                               'losses). Lower LR_EFFDET (e.g. 2e-4) and re-run this cell.')
                        continue
                    _bad = 0
                    opt.zero_grad(set_to_none=True)
                    _scaler.scale(loss).backward()
                    _scaler.unscale_(opt)            # no-op when AMP is off
                    torch.nn.utils.clip_grad_norm_(params, max_norm=10.0)
                    _scaler.step(opt); _scaler.update()
                    running += loss.detach(); nb += 1
                sched.step()
                print(f'  epoch {ep+1}/{EPOCHS_TV}  loss={float(running)/max(nb,1):.4f}  ({time.time()-_t0:.0f}s)')
            # Save into CKPT_ROOT (Drive-backed) instead of a local relative path, so this
            # checkpoint survives a full Colab runtime recycle, not just a kernel restart.
            _ckpt_dir = os.path.join(globals().get('CKPT_ROOT', 'runs_cardd'), 'torchvision')
            os.makedirs(_ckpt_dir, exist_ok=True)
            _effdet_ckpt = os.path.join(_ckpt_dir, 'efficientdet_d0.pt')
            torch.save(bench.model.state_dict(), _effdet_ckpt)
            # Memory policy: park on CPU; predict_effdet moves it to the GPU only while scoring.
            bench = bench.cpu().eval()
            effdet_model = bench
            effdet_predict_bench = DetBenchPredict(bench.model).eval()
            del opt, sched, _scaler, ld
            import gc as _gc; _gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            print('  saved', _effdet_ckpt)
        # (set for BOTH the skip path and the fresh-training path above)
        effdet_is_real = True
        EFFDET_NAME = 'EfficientDet-D0'
    except Exception as e:
        print('EfficientDet (effdet) path unavailable ->', repr(e), '\n  Falling back to RetinaNet.')
        USE_EFFICIENTDET = False

if not effdet_is_real:
    effdet_model = train_detector('retinanet', EPOCHS_TV)   # one-stage torchvision stand-in

print('One-stage comparison model trained:', EFFDET_NAME)


### C5 — ResNet50 image-level classifier (multi-label baseline)
ResNet50 classifies *which* damage types appear (no localisation). Reported separately in Part D - a deliberate contrast showing whether classification is easier than full detection.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

class CocoMultiLabel(Dataset):
    def __init__(self, split, tf):
        self.split = split
        self.coco = COCO(os.path.join(ANN_DIR, SPLIT_MAP[split][0])); self.img_dir = IMG_DIRS[split]
        self.ids = [i for i in sorted(self.coco.getImgIds())
                    if len(self.coco.getAnnIds(imgIds=[i])) > 0]
        self.tf = tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        iid = self.ids[idx]; info = self.coco.loadImgs([iid])[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        y = torch.zeros(NUM_CLASSES)
        for a in self.coco.loadAnns(self.coco.getAnnIds(imgIds=[iid])):
            y[catid_to_idx_by_split[self.split][a['category_id']]] = 1.0
        return self.tf(img), y

def train_resnet50(epochs=EPOCHS_CLS):
    tr = DataLoader(CocoMultiLabel('train', train_tf), batch_size=32, shuffle=True, num_workers=2)
    net = resnet50(weights=ResNet50_Weights.DEFAULT)
    net.fc = torch.nn.Linear(net.fc.in_features, NUM_CLASSES); net = net.to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=1e-4); crit = torch.nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        net.train(); run, nb = 0.0, 0
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = crit(net(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); run += float(loss); nb += 1
        print(f'  epoch {ep+1}/{epochs}  bce={run/max(nb,1):.4f}')
    # Save into CKPT_ROOT (Drive-backed) instead of a local relative path, so this
    # checkpoint survives a full Colab runtime recycle, not just a kernel restart.
    _ckpt_dir = os.path.join(globals().get('CKPT_ROOT', 'runs_cardd'), 'torchvision')
    os.makedirs(_ckpt_dir, exist_ok=True)
    torch.save(net.state_dict(), os.path.join(_ckpt_dir, 'resnet50_cls.pt'))
    # Memory policy: park on CPU; eval_resnet50 moves it to the GPU only while scoring.
    net = net.cpu().eval()
    del opt
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return net

# Crash-proofing: SKIP training if the classifier checkpoint already exists on Drive.
_cls_ckpt = os.path.join(globals().get('CKPT_ROOT', 'runs_cardd'), 'torchvision', 'resnet50_cls.pt')
if os.path.exists(_cls_ckpt) and not globals().get('RETRAIN_IF_CHECKPOINT', False):
    print('ResNet50 classifier: FINISHED checkpoint found -> skipping training,', _cls_ckpt)
    resnet_cls = resnet50(weights=None)
    resnet_cls.fc = torch.nn.Linear(resnet_cls.fc.in_features, NUM_CLASSES)
    resnet_cls.load_state_dict(torch.load(_cls_ckpt, map_location='cpu'))
    resnet_cls = resnet_cls.cpu().eval()
else:
    resnet_cls = train_resnet50()
print('ResNet50 classifier ready.')


### C6 — RT-DETR (hybrid CNN-Transformer: CNN backbone + transformer encoder-decoder)

RT-DETR pairs a convolutional backbone (local features) with a transformer encoder-decoder that does NMS-free, set-based prediction (global context). It ships inside Ultralytics and reuses the **same** YOLO-format `data.yaml` built in Step 13, so it drops straight into the unified evaluation harness. `fliplr` + `hsv_v` apply the proposal's flip / brightness augmentation.


In [ ]:
# Self-heal: guarantee ultralytics is present even if cell 2 (or
# ensure_packages) never ran in this runtime (e.g. after a Colab
# disconnect/reconnect mid-session). Fully standalone - does not
# depend on any earlier cell having executed.
import importlib as _importlib, subprocess as _subprocess, sys as _sys
try:
    _importlib.import_module('ultralytics')
except ImportError:
    print('ultralytics missing in this runtime - installing ultralytics now...')
    _subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    print('Done.')

# C6 — RT-DETR (real hybrid CNN-Transformer). No extra install: bundled with ultralytics.
# Reuses the same YOLO-format dataset (yaml_path) and the same Results API as C1 (YOLOv8),
# so predictions feed the identical evaluate()/confusion-matrix machinery used for every model.
from ultralytics import RTDETR

# DETR-family detectors converge more slowly than YOLO, so RT-DETR gets more epochs than the
# torchvision models. 40 is the full final-numbers budget (this is the longest cell in the
# notebook, several hours on a T4) — but it resumes from its last epoch after any disconnect,
# so it is safe to let it span two sessions. Drop to ~12 only for a quick pipeline check.
EPOCHS_RTDETR = 40
BATCH_RTDETR  = max(2, BATCH_YOLO // 2)   # transformer decoder is memory-heavy; halve YOLO batch

# Train into CKPT_ROOT (Drive-backed) instead of a local relative path, so best.pt
# survives a full Colab runtime recycle, not just a kernel restart.
_rtdetr_project = globals().get('CKPT_ROOT', 'runs_cardd')
_rt_w = os.path.join(_rtdetr_project, 'rtdetr-l', 'weights')
_rt_last, _rt_best = os.path.join(_rt_w, 'last.pt'), os.path.join(_rt_w, 'best.pt')

# Crash-proofing: SKIP if finished, RESUME if interrupted (this is the cell that previously
# crashed after a full day — a disconnect now costs at most one epoch, not the whole run).
if (os.path.exists(_rt_best) and os.path.exists(_rt_last)
        and ultralytics_run_finished(_rt_last) and not RETRAIN_IF_CHECKPOINT):
    print('RT-DETR: FINISHED run found on Drive -> skipping training, using', _rt_best)
elif os.path.exists(_rt_last) and not RETRAIN_IF_CHECKPOINT:
    print('RT-DETR: INTERRUPTED run found -> resuming from', _rt_last)
    rtdetr = RTDETR(_rt_last)
    try:
        rtdetr.train(resume=True)
    except Exception as _e:
        if os.path.exists(_rt_best):
            print('RT-DETR: resume not needed/possible ->', _e, '\n  Using existing', _rt_best)
        else:
            raise
else:
    rtdetr = RTDETR('rtdetr-l.pt')        # pretrained weights -> transfer learning
    rtdetr.train(data=yaml_path, epochs=EPOCHS_RTDETR, imgsz=IMGSZ, batch=BATCH_RTDETR,
                 device=(0 if DEVICE=='cuda' else 'cpu'), project=_rtdetr_project, name='rtdetr-l',
                 exist_ok=True, seed=SEED, fliplr=0.5, hsv_v=0.4)

# Resolve best.pt robustly (same approach as the YOLOv8 cell).
RTDETR_BEST = None
try:
    RTDETR_BEST = str(rtdetr.trainer.best)
except Exception:
    pass
if not (RTDETR_BEST and os.path.exists(RTDETR_BEST)):
    alt = (sorted(glob.glob(os.path.join(_rtdetr_project, 'rtdetr-l*/weights/best.pt'))) or
           sorted(glob.glob('runs_cardd/rtdetr-l*/weights/best.pt')) or
           sorted(glob.glob('**/rtdetr*/weights/best.pt', recursive=True)) or
           sorted(glob.glob('**/rtdetr*/weights/last.pt', recursive=True)))
    RTDETR_BEST = alt[-1] if alt else RTDETR_BEST
assert RTDETR_BEST and os.path.exists(RTDETR_BEST), (
    'RT-DETR weights not found - C6 training did not complete in this session. '
    'Re-run this RT-DETR cell to the end before running evaluation.')
print('RT-DETR done ->', RTDETR_BEST)
# Free the GPU: predict_rtdetr reloads from RTDETR_BEST on disk, so drop the live object.
if 'rtdetr' in globals(): del rtdetr
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## Part D — Evaluate & compare  (Objective 2.5)

One evaluator scores every detector identically: **mAP@50, mAP@50:95** (torchmetrics), **Precision/Recall/F1** (greedy IoU matching at IoU=0.5, conf=0.25), **inference time** (ms/img + FPS), and a **confusion matrix**.

**Graceful scoring:** the evaluation cell scores every model whose finished checkpoint exists and *lists* the ones that are still missing instead of stopping — so even a partial training run gives you a real comparison table for the report. Finish the missing models later and re-run the evaluation cell to complete the table.


In [ ]:
# Self-heal: guarantee torchmetrics is present even if cell 2 (or
# ensure_packages) never ran in this runtime (e.g. after a Colab
# disconnect/reconnect mid-session). Fully standalone - does not
# depend on any earlier cell having executed.
import importlib as _importlib, subprocess as _subprocess, sys as _sys
try:
    _importlib.import_module('torchmetrics')
except ImportError:
    print('torchmetrics missing in this runtime - installing torchmetrics now...')
    _subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics'], check=True)
    print('Done.')

from torchmetrics.detection.mean_ap import MeanAveragePrecision
CONF_THR, IOU_THR = 0.25, 0.5

def box_iou_np(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    area_a = (a[:,2]-a[:,0])*(a[:,3]-a[:,1]); area_b = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    iou = np.zeros((len(a), len(b)))
    for i in range(len(a)):
        xx1 = np.maximum(a[i,0], b[:,0]); yy1 = np.maximum(a[i,1], b[:,1])
        xx2 = np.minimum(a[i,2], b[:,2]); yy2 = np.minimum(a[i,3], b[:,3])
        inter = np.clip(xx2-xx1, 0, None) * np.clip(yy2-yy1, 0, None)
        iou[i] = inter / (area_a[i] + area_b - inter + 1e-9)
    return iou

def evaluate(preds, gts, model_name, infer_ms):
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    tm_p = [{'boxes': torch.tensor(p['boxes'], dtype=torch.float32) if len(p['boxes']) else torch.zeros((0,4)),
             'scores': torch.tensor(p['scores'], dtype=torch.float32) if len(p['boxes']) else torch.zeros((0,)),
             'labels': torch.tensor(p['labels'], dtype=torch.int64) if len(p['boxes']) else torch.zeros((0,), dtype=torch.int64)} for p in preds]
    tm_g = [{'boxes': torch.tensor(g['boxes'], dtype=torch.float32) if len(g['boxes']) else torch.zeros((0,4)),
             'labels': torch.tensor(g['labels'], dtype=torch.int64) if len(g['boxes']) else torch.zeros((0,), dtype=torch.int64)} for g in gts]
    metric.update(tm_p, tm_g); mres = metric.compute()
    map50, map5095 = float(mres['map_50']), float(mres['map'])

    C = NUM_CLASSES; cm = np.zeros((C+1, C+1), dtype=int); TP = FP = FN = 0
    for p, g in zip(preds, gts):
        pb = np.array(p['boxes']).reshape(-1,4); pl = np.array(p['labels']).reshape(-1)
        ps = np.array(p['scores']).reshape(-1) if len(p['boxes']) else np.zeros(0)
        keep = ps >= CONF_THR; pb, pl, ps = pb[keep], pl[keep], ps[keep]
        order = np.argsort(-ps); pb, pl = pb[order], pl[order]
        gb = np.array(g['boxes']).reshape(-1,4); gl = np.array(g['labels']).reshape(-1)
        matched = np.zeros(len(gb), dtype=bool); iou = box_iou_np(pb, gb)
        for i in range(len(pb)):
            j, best = -1, IOU_THR
            for jj in range(len(gb)):
                if not matched[jj] and iou[i, jj] >= best: best, j = iou[i, jj], jj
            if j >= 0:
                matched[j] = True; cm[gl[j], pl[i]] += 1
                TP += int(gl[j] == pl[i]); FP += int(gl[j] != pl[i])
            else:
                FP += 1; cm[C, pl[i]] += 1
        for jj in range(len(gb)):
            if not matched[jj]: FN += 1; cm[gl[jj], C] += 1
    prec = TP/(TP+FP+1e-9); rec = TP/(TP+FN+1e-9); f1 = 2*prec*rec/(prec+rec+1e-9)
    row = {'Model': model_name, 'mAP@50': round(map50,4), 'mAP@50:95': round(map5095,4),
           'Precision': round(prec,4), 'Recall': round(rec,4), 'F1': round(f1,4),
           'ms/img': round(infer_ms,2), 'FPS': round(1000/infer_ms,1) if infer_ms>0 else 0}
    return row, cm


In [ ]:
val_ds = CocoDetDataset('val', train_aug=False)
val_paths = [os.path.join(val_ds.img_dir, val_ds.coco.loadImgs([i])[0]['file_name']) for i in val_ds.ids]
# Build GT directly from the annotation file. The old version called val_ds[i], which decoded
# every full-resolution validation image from disk just to read its box coordinates — several
# minutes of pointless JPEG decoding. _clipped_boxes() returns the identical boxes/labels
# (train_aug=False, so __getitem__ applied no transform to them anyway) with zero image I/O.
GT = []
for iid in val_ds.ids:
    _b, _l = val_ds._clipped_boxes(iid)
    GT.append({'boxes': np.array(_b, dtype=np.float32).reshape(-1, 4),
               'labels': np.array(_l, dtype=np.int64) - 1})
print('Validation images for scoring:', len(GT))


In [ ]:
EVAL_BATCH = 8   # images per forward pass during evaluation; lower to 4 if you ever OOM here

@torch.no_grad()
def predict_torchvision(model, batch=None):
    """Batched inference: N images per forward pass instead of one — far fewer GPU round-trips.
    ms/img is measured over model forward time only (image loading excluded), same as before."""
    batch = EVAL_BATCH if batch is None else batch
    # Models are parked on CPU between uses (memory policy). Borrow the GPU for
    # this evaluation only, and hand it back at the end so the next model fits.
    model = model.to(DEVICE).eval(); preds, total_t, n = [], 0.0, 0
    for c0 in range(0, len(val_paths), batch):
        chunk = val_paths[c0:c0 + batch]
        imgs = [F.to_tensor(Image.open(p).convert('RGB')).to(DEVICE) for p in chunk]
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.time(); outs = model(imgs)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        total_t += (time.time() - t0) * 1000; n += len(chunk)
        for out in outs:
            preds.append({'boxes': out['boxes'].cpu().numpy(), 'scores': out['scores'].cpu().numpy(),
                          'labels': (out['labels'].cpu().numpy() - 1)})
    model.cpu()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, total_t / max(n, 1)

@torch.no_grad()
def predict_yolo(weights):
    if not os.path.exists(weights):
        raise FileNotFoundError(f'YOLO weights not found at {weights}. Re-run the C1 YOLOv8 '
                                f'training cell to completion before evaluating.')
    # Chunked calls: one predict() per 16 images instead of per image — Ultralytics has real
    # per-call setup overhead, so this alone is a large speedup. max_det=100 is the COCO-standard
    # cap; without it, conf=0.001 floods the mAP computation with thousands of junk boxes.
    det = YOLO(weights); preds, total_t, n = [], 0.0, 0
    for c0 in range(0, len(val_paths), 16):
        chunk = val_paths[c0:c0 + 16]
        rs = det.predict(chunk, imgsz=IMGSZ, conf=0.001, max_det=100,
                         device=(0 if DEVICE=='cuda' else 'cpu'), verbose=False)
        for r in rs:
            total_t += float(r.speed.get('inference', 0.0)); n += 1
            if r.boxes is None or len(r.boxes) == 0:
                preds.append({'boxes': np.zeros((0,4)), 'scores': np.zeros(0), 'labels': np.zeros(0, int)})
            else:
                preds.append({'boxes': r.boxes.xyxy.cpu().numpy(), 'scores': r.boxes.conf.cpu().numpy(),
                              'labels': r.boxes.cls.cpu().numpy().astype(int)})
    del det
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, total_t/max(n,1)


@torch.no_grad()
def predict_effdet(predict_bench, batch=None):
    """Run real EfficientDet over the SAME val images as GT, un-letterboxing detections
    back to original coords. Batched (fixed 512x512 input makes stacking trivial).
    Returns preds (0-based labels) + mean ms/img, matching the other predictors."""
    batch = EVAL_BATCH if batch is None else batch
    predict_bench = predict_bench.to(DEVICE).eval(); preds, total_t, n = [], 0.0, 0
    for c0 in range(0, len(val_paths), batch):
        chunk = val_paths[c0:c0 + batch]
        metas, tens = [], []
        for path in chunk:
            t, s, pad_l, pad_t, W, H = _letterbox(Image.open(path).convert('RGB'))
            tens.append(t); metas.append((s, pad_l, pad_t, W, H))
        x = torch.stack(tens).to(DEVICE)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.time(); dets = predict_bench(x).cpu().numpy()   # [B, max_det, 6]: x1,y1,x2,y2,score,class(1-based)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        total_t += (time.time() - t0) * 1000; n += len(chunk)
        for det, (s, pad_l, pad_t, W, H) in zip(dets, metas):
            if det.shape[0] == 0:
                preds.append({'boxes': np.zeros((0, 4)), 'scores': np.zeros(0), 'labels': np.zeros(0, int)}); continue
            boxes = _unletterbox(det[:, :4], s, pad_l, pad_t, W, H)
            preds.append({'boxes': boxes, 'scores': det[:, 4],
                          'labels': (det[:, 5].astype(int) - 1)})        # 1-based -> 0-based for eval
    predict_bench.cpu()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, total_t / max(n, 1)


In [ ]:
@torch.no_grad()
def predict_rtdetr(weights):
    '''RT-DETR shares the Ultralytics Results API with YOLO, so this mirrors predict_yolo
    (chunked calls + COCO-standard max_det=100 cap). Labels are already 0-based (trained on
    the same YOLO-format labels) -> matches GT/eval.'''
    if not os.path.exists(weights):
        raise FileNotFoundError(f'RT-DETR weights not found at {weights}. Re-run the C6 cell first.')
    det = RTDETR(weights); preds, total_t, n = [], 0.0, 0
    for c0 in range(0, len(val_paths), 16):
        chunk = val_paths[c0:c0 + 16]
        rs = det.predict(chunk, imgsz=IMGSZ, conf=0.001, max_det=100,
                         device=(0 if DEVICE=='cuda' else 'cpu'), verbose=False)
        for r in rs:
            total_t += float(r.speed.get('inference', 0.0)); n += 1
            if r.boxes is None or len(r.boxes) == 0:
                preds.append({'boxes': np.zeros((0,4)), 'scores': np.zeros(0), 'labels': np.zeros(0, int)})
            else:
                preds.append({'boxes': r.boxes.xyxy.cpu().numpy(), 'scores': r.boxes.conf.cpu().numpy(),
                              'labels': r.boxes.cls.cpu().numpy().astype(int)})
    del det
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, total_t/max(n,1)


In [ ]:
# Self-heal + GRACEFUL evaluation: reload whatever finished checkpoints exist and score
# ONLY those models. A missing or unfinished model no longer kills Part D - it is listed
# at the end with the exact cell to re-run. Even ONE finished model gives you a table.
import glob as _glob
import time, numpy as np, pandas as pd, torch
from ultralytics import YOLO, RTDETR

def _ckpt_roots():
    roots = []
    if 'CKPT_ROOT' in globals():
        roots.append(CKPT_ROOT)
    roots.append('runs_cardd')  # legacy local-disk location
    return roots

def _find_ckpt(*rel_patterns):
    for root in _ckpt_roots():
        for rel in rel_patterns:
            hits = sorted(_glob.glob(os.path.join(root, rel), recursive=True))
            if hits:
                return hits[-1]
    return None

def _finite_ok(m):
    f = globals().get('_params_finite')
    return True if f is None else f(m)

available, missing = {}, []      # display name -> predictor spec ; [(name, how to fix)]

# --- YOLOv8 ---
if not ('YOLO_BEST' in globals() and YOLO_BEST and os.path.exists(YOLO_BEST)):
    YOLO_BEST = _find_ckpt('yolov8s*/weights/best.pt', '**/yolov8s*/weights/best.pt')
if YOLO_BEST and os.path.exists(YOLO_BEST):
    available['YOLOv8s (CNN, 1-stage)'] = ('ul', YOLO, YOLO_BEST)
    print('YOLOv8s ready ->', YOLO_BEST)
else:
    missing.append(('YOLOv8s', 'run C1 to the end'))

# --- RT-DETR ---
if not ('RTDETR_BEST' in globals() and RTDETR_BEST and os.path.exists(RTDETR_BEST)):
    RTDETR_BEST = _find_ckpt('rtdetr-l*/weights/best.pt', '**/rtdetr*/weights/best.pt')
if RTDETR_BEST and os.path.exists(RTDETR_BEST):
    available['RT-DETR (hybrid CNN-Transformer)'] = ('ul', RTDETR, RTDETR_BEST)
    print('RT-DETR ready ->', RTDETR_BEST)
else:
    missing.append(('RT-DETR', 'run C6 to the end'))

# --- Faster R-CNN ---
if 'frcnn' not in globals():
    _c = _find_ckpt('torchvision/fasterrcnn.pt')
    if _c:
        frcnn = build_detector('fasterrcnn', NUM_CLASSES + 1)
        frcnn.load_state_dict(torch.load(_c, map_location='cpu')); frcnn.eval()
        print('Recovered frcnn from', _c)
if 'frcnn' in globals() and _finite_ok(frcnn):
    available['Faster R-CNN (2-stage)'] = ('tv', frcnn)
else:
    missing.append(('Faster R-CNN', 'run C2 to the end'))

# --- SSD ---
if 'ssd' not in globals():
    _c = _find_ckpt('torchvision/ssd.pt')
    if _c:
        ssd = build_detector('ssd', NUM_CLASSES + 1)
        ssd.load_state_dict(torch.load(_c, map_location='cpu')); ssd.eval()
        print('Recovered ssd from', _c)
if 'ssd' in globals() and _finite_ok(ssd):
    available['SSD (1-stage)'] = ('tv', ssd)
else:
    missing.append(('SSD', 'run C3 to the end (a NaN/diverged old checkpoint also lands here)'))

# --- EfficientDet-D0 (real) or RetinaNet stand-in ---
if 'effdet_model' not in globals() or effdet_model is None:
    _real_ckpt = _find_ckpt('torchvision/efficientdet_d0.pt')
    _fallback_ckpt = _find_ckpt('torchvision/retinanet.pt')
    _effdet_img_size = globals().get('EFFDET_IMG_SIZE', 512)
    if _real_ckpt:
        from effdet import create_model, DetBenchPredict
        _bench = create_model('tf_efficientdet_d0', bench_task='train', num_classes=NUM_CLASSES,
                              pretrained=False, pretrained_backbone=False, bench_labeler=True,
                              image_size=(_effdet_img_size, _effdet_img_size))       # CPU
        _bench.model.load_state_dict(torch.load(_real_ckpt, map_location='cpu'))
        effdet_predict_bench = DetBenchPredict(_bench.model).eval()
        effdet_model = effdet_predict_bench
        effdet_is_real = True
        EFFDET_NAME = 'EfficientDet-D0'
        print('Recovered EfficientDet-D0 from', _real_ckpt)
    elif _fallback_ckpt:
        effdet_model = build_detector('retinanet', NUM_CLASSES + 1)
        effdet_model.load_state_dict(torch.load(_fallback_ckpt, map_location='cpu'))
        effdet_model.eval()
        effdet_is_real = False
        EFFDET_NAME = 'RetinaNet (EfficientDet stand-in)'
        print('Recovered RetinaNet fallback from', _fallback_ckpt)
if 'effdet_model' in globals() and effdet_model is not None:
    available[EFFDET_NAME] = ('ed', effdet_predict_bench) if effdet_is_real else ('tv', effdet_model)
else:
    missing.append(('EfficientDet/RetinaNet', 'run C4 to the end'))

if not available:
    raise RuntimeError('No finished model checkpoints found at all. Run Part C first - '
                       'even ONE finished model is enough to get this table.')

# ---------------- FAST EVALUATION ----------------
EVAL_MAX_IMAGES = None       # FULL validation set -> final report numbers (set to e.g. 200 for a quick draft)
_vp, _gt = val_paths, GT
if EVAL_MAX_IMAGES and len(_vp) > EVAL_MAX_IMAGES:
    _keep = np.linspace(0, len(_vp) - 1, EVAL_MAX_IMAGES).astype(int)
    _vp = [_vp[i] for i in _keep]; _gt = [_gt[i] for i in _keep]
    print(f'QUICK MODE: scoring on {len(_gt)} evenly-spaced val images')

EVAL_BATCH = 8   # lower to 4 if you OOM here

def _hb(done, total, t0):
    r = done / max(time.time() - t0, 1e-9)
    print(f'    {done}/{total} imgs ({r:.1f} img/s, ~{(total-done)/max(r,1e-9):.0f}s left)', flush=True)

@torch.no_grad()
def _pred_tv(model, batch=None):
    batch = batch or EVAL_BATCH
    model = model.to(DEVICE).eval(); preds, tt, n = [], 0.0, 0
    t0, mark = time.time(), max(1, len(_vp)//4)
    for c0 in range(0, len(_vp), batch):
        if c0 and c0 % ((mark//batch+1)*batch) == 0: _hb(c0, len(_vp), t0)
        imgs = [F.to_tensor(Image.open(p).convert('RGB')).to(DEVICE) for p in _vp[c0:c0+batch]]
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t1 = time.time(); outs = model(imgs)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        tt += (time.time()-t1)*1000; n += len(imgs)
        for o in outs:
            preds.append({'boxes': o['boxes'].cpu().numpy(), 'scores': o['scores'].cpu().numpy(),
                          'labels': o['labels'].cpu().numpy() - 1})
    model.cpu()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, tt/max(n,1)

@torch.no_grad()
def _pred_ul(ctor, weights):
    det = ctor(weights); preds, tt, n = [], 0.0, 0
    t0, mark = time.time(), max(1, len(_vp)//4)
    for c0 in range(0, len(_vp), 16):
        if c0 and c0 % ((mark//16+1)*16) == 0: _hb(c0, len(_vp), t0)
        for r in det.predict(_vp[c0:c0+16], imgsz=IMGSZ, conf=0.001, max_det=100,
                             device=(0 if DEVICE=='cuda' else 'cpu'), verbose=False):
            tt += float(r.speed.get('inference', 0.0)); n += 1
            if r.boxes is None or len(r.boxes) == 0:
                preds.append({'boxes': np.zeros((0,4)), 'scores': np.zeros(0), 'labels': np.zeros(0, int)})
            else:
                preds.append({'boxes': r.boxes.xyxy.cpu().numpy(), 'scores': r.boxes.conf.cpu().numpy(),
                              'labels': r.boxes.cls.cpu().numpy().astype(int)})
    del det
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, tt/max(n,1)

@torch.no_grad()
def _pred_ed(bench, batch=None):
    batch = batch or EVAL_BATCH
    bench = bench.to(DEVICE).eval(); preds, tt, n = [], 0.0, 0
    t0, mark = time.time(), max(1, len(_vp)//4)
    for c0 in range(0, len(_vp), batch):
        if c0 and c0 % ((mark//batch+1)*batch) == 0: _hb(c0, len(_vp), t0)
        metas, tens = [], []
        for p in _vp[c0:c0+batch]:
            t_, s, pl, pt, W, H = _letterbox(Image.open(p).convert('RGB'))
            tens.append(t_); metas.append((s, pl, pt, W, H))
        x = torch.stack(tens).to(DEVICE)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t1 = time.time(); dets = bench(x).cpu().numpy()
        if DEVICE == 'cuda': torch.cuda.synchronize()
        tt += (time.time()-t1)*1000; n += len(tens)
        for det, (s, pl, pt, W, H) in zip(dets, metas):
            if det.shape[0] == 0:
                preds.append({'boxes': np.zeros((0,4)), 'scores': np.zeros(0), 'labels': np.zeros(0, int)}); continue
            preds.append({'boxes': _unletterbox(det[:, :4], s, pl, pt, W, H),
                          'scores': det[:, 4], 'labels': det[:, 5].astype(int) - 1})
    bench.cpu()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return preds, tt/max(n,1)

print(f'Scoring {len(available)} model(s) on', len(_gt), 'val images | device:', DEVICE)
if DEVICE == 'cpu':
    print('*** WARNING: NO GPU (Colab quota likely ran out) -> ~50x slower.')
    print('*** Fix: Runtime > Change runtime type > T4 GPU. Checkpoints are safe on Drive.')

results, confusions = [], {}
_short = {'YOLOv8s (CNN, 1-stage)': 'YOLOv8s', 'Faster R-CNN (2-stage)': 'FasterRCNN',
          'SSD (1-stage)': 'SSD', 'RT-DETR (hybrid CNN-Transformer)': 'RT-DETR'}
for _i, (_name, _spec) in enumerate(available.items(), 1):
    try:
        if _spec[0] == 'ul':
            _p, _t = _pred_ul(_spec[1], _spec[2])
        elif _spec[0] == 'tv':
            _p, _t = _pred_tv(_spec[1])
        else:
            _p, _t = _pred_ed(_spec[1])
        r, cm = evaluate(_p, _gt, _name, _t)
        results.append(r); confusions[_short.get(_name, 'EfficientDet')] = cm
        print(f'  [{_i}/{len(available)}] {_name} done')
    except Exception as _e:
        print(f'  [{_i}/{len(available)}] {_name} FAILED during inference -> {_e!r}  (skipped)')
        missing.append((_name, 'inference failed - see the error printed above'))

if missing:
    print('\nNOT scored (finish these, then re-run THIS cell to add them to the table):')
    for _n, _fix in missing:
        print(f'   - {_n}: {_fix}')

if not results:
    raise RuntimeError('Every available model failed during inference - see the errors above.')

comparison_df = pd.DataFrame(results)
print('UNIFIED MODEL COMPARISON (real, identical val set)'); print('='*90)
print(comparison_df.to_string(index=False))
comparison_df.to_csv('model_comparison.csv', index=False)
print('\nSaved: model_comparison.csv')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 6))
mp = ['mAP@50', 'mAP@50:95', 'Precision', 'Recall', 'F1']
x = np.arange(len(mp)); w = 0.8/max(len(comparison_df),1)
for i, (_, r) in enumerate(comparison_df.iterrows()):
    ax[0].bar(x + i*w, [r[m] for m in mp], w, label=r['Model'])
ax[0].set_xticks(x + w*(len(comparison_df)-1)/2); ax[0].set_xticklabels(mp, rotation=15)
ax[0].set_title('Accuracy metrics', fontweight='bold'); ax[0].legend(fontsize=8); ax[0].grid(axis='y', alpha=0.3)
ax[1].bar(comparison_df['Model'], comparison_df['FPS'], color=sns.color_palette('mako', len(comparison_df)))
ax[1].set_title('Speed (FPS, higher = faster)', fontweight='bold'); ax[1].tick_params(axis='x', rotation=20); ax[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=200, bbox_inches='tight'); plt.show()
print('Saved: model_comparison.png')


In [ ]:
if 'confusions' in globals() and confusions:
    labels_axis = CLASS_NAMES + ['background']; n = len(confusions)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6)); axes = np.array(axes).reshape(-1)
    for ax, (name, cm) in zip(axes, confusions.items()):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=labels_axis, yticklabels=labels_axis, ax=ax)
        ax.set_title(name, fontweight='bold'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=45); ax.tick_params(axis='y', rotation=0)
    plt.tight_layout(); plt.savefig('confusion_matrices.png', dpi=200, bbox_inches='tight'); plt.show()
    print('Saved: confusion_matrices.png')

else:
    print('No confusion matrices yet - run the evaluation cell after at least one model finishes.')


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

# Self-heal + graceful: reload resnet_cls from its checkpoint if possible; if the
# classifier was never trained, SKIP this table with a clear message instead of stopping.
if 'resnet_cls' not in globals():
    _ckpt = None
    for _root in ([CKPT_ROOT] if 'CKPT_ROOT' in globals() else []) + ['runs_cardd']:
        _cand = os.path.join(_root, 'torchvision', 'resnet50_cls.pt')
        if os.path.exists(_cand):
            _ckpt = _cand
            break
    if _ckpt:
        resnet_cls = resnet50(weights=None)
        resnet_cls.fc = torch.nn.Linear(resnet_cls.fc.in_features, NUM_CLASSES)
        resnet_cls.load_state_dict(torch.load(_ckpt, map_location='cpu'))
        resnet_cls = resnet_cls.eval()   # stays on CPU; eval_resnet50 borrows the GPU
        print('Recovered resnet_cls from', _ckpt)

if 'resnet_cls' not in globals():
    print('ResNet50 classifier not trained yet -> skipping this table. Run C5, then re-run this cell.')
else:
    from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
    @torch.no_grad()
    def eval_resnet50(net, thr=0.5):
        net = net.to(DEVICE).eval()
        ld = DataLoader(CocoMultiLabel('val', eval_tf), batch_size=32, num_workers=2)
        P, Y = [], []
        for x, y in ld:
            P.append(torch.sigmoid(net(x.to(DEVICE))).cpu().numpy()); Y.append(y.numpy())
        net.cpu()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        P, Y = np.vstack(P), np.vstack(Y); B = (P >= thr).astype(int)
        return {'Model': 'ResNet50 (classification baseline)',
                'Micro-F1': round(f1_score(Y, B, average='micro', zero_division=0), 4),
                'Macro-F1': round(f1_score(Y, B, average='macro', zero_division=0), 4),
                'Precision': round(precision_score(Y, B, average='micro', zero_division=0), 4),
                'Recall': round(recall_score(Y, B, average='micro', zero_division=0), 4),
                'Subset-Acc': round(accuracy_score(Y, B), 4)}

    resnet_metrics = eval_resnet50(resnet_cls)
    print('RESNET50 CLASSIFICATION BASELINE'); print('='*55)
    for k, v in resnet_metrics.items(): print(f'   {k:<12}: {v}')
    pd.DataFrame([resnet_metrics]).to_csv('resnet50_classification.csv', index=False)
    print('Saved: resnet50_classification.csv')


## Part E — Explainability  (Objective 2.5)
**Grad-CAM** (true gradient-based) on ResNet50, and **EigenCAM** on YOLOv8 (Grad-CAM needs a single class target, awkward for multi-box detectors - state this choice in your write-up).


In [ ]:
# Self-heal: guarantee pytorch_grad_cam is present even if cell 2 (or
# ensure_packages) never ran in this runtime (e.g. after a Colab
# disconnect/reconnect mid-session). Fully standalone - does not
# depend on any earlier cell having executed.
import importlib as _importlib, subprocess as _subprocess, sys as _sys
try:
    _importlib.import_module('pytorch_grad_cam')
except ImportError:
    print('pytorch_grad_cam missing in this runtime - installing grad-cam now...')
    _subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'grad-cam'], check=True)
    print('Done.')

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

imgs = sorted(glob.glob(os.path.join(YOLO_ROOT, 'images', 'val', '*')))[:3]
if imgs:
    # Memory policy: resnet_cls is parked on CPU between uses. Borrow the GPU for
    # Grad-CAM (inputs below are .to(DEVICE), so the model must be there too).
    resnet_cls = resnet_cls.to(DEVICE).eval()
    cam = GradCAM(model=resnet_cls, target_layers=[resnet_cls.layer4[-1]])
    fig, axes = plt.subplots(len(imgs), 2, figsize=(12, 5*len(imgs))); axes = np.array(axes).reshape(len(imgs), 2)
    for k, ip in enumerate(imgs):
        rgb = np.array(Image.open(ip).convert('RGB').resize((224,224)))/255.0
        x = eval_tf(Image.open(ip).convert('RGB')).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): top = int(torch.sigmoid(resnet_cls(x))[0].argmax())
        g = cam(input_tensor=x, targets=[ClassifierOutputTarget(top)])[0]
        vis = show_cam_on_image(rgb.astype(np.float32), g, use_rgb=True)
        axes[k,0].imshow(rgb); axes[k,0].set_title('Input'); axes[k,0].axis('off')
        axes[k,1].imshow(vis); axes[k,1].set_title(f'Grad-CAM ({CLASS_NAMES[top]})'); axes[k,1].axis('off')
    plt.tight_layout(); plt.savefig('gradcam_resnet50.png', dpi=200, bbox_inches='tight'); plt.show()
    print('Saved: gradcam_resnet50.png')
    resnet_cls = resnet_cls.cpu()          # park it again
    if DEVICE == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Self-heal: guarantee pytorch_grad_cam is present even if cell 2 (or
# ensure_packages) never ran in this runtime (e.g. after a Colab
# disconnect/reconnect mid-session). Fully standalone - does not
# depend on any earlier cell having executed.
import importlib as _importlib, subprocess as _subprocess, sys as _sys
try:
    _importlib.import_module('pytorch_grad_cam')
except ImportError:
    print('pytorch_grad_cam missing in this runtime - installing grad-cam now...')
    _subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'grad-cam'], check=True)
    print('Done.')

from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image   # standalone: don't rely on the previous cell
imgs = sorted(glob.glob(os.path.join(YOLO_ROOT, 'images', 'val', '*')))
if imgs:
    rgb = np.array(Image.open(imgs[0]).convert('RGB').resize((IMGSZ, IMGSZ)))/255.0
    tensor = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).float()
    if DEVICE == 'cuda': tensor = tensor.cuda()
    # The input tensor above is moved to CUDA, so the model must live there too —
    # YOLO(weights) loads on CPU by default, and the old device mismatch was being
    # swallowed by the except below (symptom: 'EigenCAM error' instead of a figure).
    ym = YOLO(YOLO_BEST).model.to(DEVICE).eval()
    class _RawOut(torch.nn.Module):
        '''YOLOv8's eval forward returns a (preds, feats) TUPLE; grad-cam's BaseCAM then calls
        outputs.cpu() on it and crashes - the old symptom was this cell printing 'EigenCAM
        error' instead of a figure. Returning a plain 2-D tensor keeps the library's
        bookkeeping happy; EigenCAM itself only uses the hooked layer activations, so the
        CAM is unchanged.'''
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x):
            o = self.m(x)
            o = o[0] if isinstance(o, (list, tuple)) else o
            return o.flatten(1).float()
    try:
        g = EigenCAM(_RawOut(ym), [ym.model[-2]])(tensor)[0]
        vis = show_cam_on_image(rgb.astype(np.float32), g, use_rgb=True)
        fig, ax = plt.subplots(1, 2, figsize=(14, 7))
        ax[0].imshow(rgb); ax[0].set_title('Input'); ax[0].axis('off')
        ax[1].imshow(vis); ax[1].set_title('EigenCAM (YOLOv8)'); ax[1].axis('off')
        plt.tight_layout(); plt.savefig('eigencam_yolov8.png', dpi=200, bbox_inches='tight'); plt.show()
        print('Saved: eigencam_yolov8.png')
    except Exception as e:
        print('EigenCAM error:', e, '\nInspect YOLO(YOLO_BEST).model.model and pick a late conv layer.')
    del ym
    if DEVICE == 'cuda': torch.cuda.empty_cache()


## Part F — Inference demo (real detections)

In [ ]:
# Self-heal + graceful: needs Part D results (comparison_df or its csv) and the YOLO
# weights. If either is missing it says exactly what to run instead of stopping the notebook.
if 'comparison_df' not in globals():
    import pandas as pd
    if os.path.exists('model_comparison.csv'):
        comparison_df = pd.read_csv('model_comparison.csv')
        print('Recovered comparison_df from model_comparison.csv (kernel state was reset).')

if 'comparison_df' not in globals() or len(comparison_df) == 0:
    print('No evaluation results yet -> run Part D first, then re-run this cell.')
elif not ('YOLO_BEST' in globals() and YOLO_BEST and os.path.exists(YOLO_BEST)):
    print('YOLO weights not found -> finish C1, then re-run this cell for the demo grid.')
else:
    best_name = comparison_df.sort_values('mAP@50:95', ascending=False).iloc[0]['Model']
    print('Best detector by mAP@50:95:', best_name)
    det = YOLO(YOLO_BEST)
    imgs = sorted(glob.glob(os.path.join(YOLO_ROOT, 'images', 'test', '*'))) \
           or sorted(glob.glob(os.path.join(YOLO_ROOT, 'images', 'val', '*')))
    sample = imgs[:6]
    res = det.predict(sample, imgsz=IMGSZ, conf=0.25, device=(0 if DEVICE=='cuda' else 'cpu'), verbose=False)
    rows = (len(sample)+2)//3
    fig, axes = plt.subplots(rows, 3, figsize=(18, 5*rows)); axes = np.array(axes).reshape(-1)
    for ax, r in zip(axes, res):
        ax.imshow(r.plot()[:, :, ::-1]); ax.axis('off')
        ax.set_title(f'{os.path.basename(r.path)} ({len(r.boxes)} det)', fontsize=9)
    for ax in axes[len(sample):]: ax.axis('off')
    plt.tight_layout(); plt.savefig('inference_demo.png', dpi=200, bbox_inches='tight'); plt.show()
    print('Saved: inference_demo.png')

## Outputs (all real)

`integrity_report.json`, `CarDD_clean/` (cleaned annotations), `class_distribution.png`, `dimensions_and_bboxes.png`, `samples_train.png`, `preprocess_resize_normalise.png`, `augmentation_demo.png`, `class_weights.json`, `model_comparison.csv`/`.png`, `confusion_matrices.png`, `resnet50_classification.csv`, `gradcam_resnet50.png`, `eigencam_yolov8.png`, `inference_demo.png`, `runs_cardd/` (weights + logs).

**Map each to an objective:** Steps 1-9 -> Obj 2.2 cleaning/EDA; Steps 10-15 -> Obj 2.2 preprocessing; C1 -> Obj 2.3; C2-C5 -> Obj 2.4; Parts D/E -> Obj 2.5. Report printed numbers exactly, explain weak classes with the Step 5 imbalance figure, and be honest about the EfficientDet stand-in.
